# Categorical BMI: three-group hierarchical model

The secondary v2.2c model fits normal weight, overweight and obesity using all eligible profiles.

Set BASE, PCA_DIR_OVERRIDE and MODEL_ROOT to the input and new output directories. Set NOTEBOOK_PATH_OVERRIDE to the saved path of this curated notebook. Select RUN_SPEC, PROFILE and WORKFLOW_STAGE. Execute installation, mounting and configuration, then the cells for the selected stage. Run the patient-influence audit only with a loaded posterior trace and completed PPC.


In [ ]:
# 1. Install the validated sampling environment
# Run first in a fresh CPU High-RAM Colab runtime, before importing PyMC.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pymc==5.28.5",
    "arviz==0.23.4",
    "nutpie==0.16.11",
])
print("Pinned Bayesian environment installed.")


In [ ]:
# 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# 3. Configuration, categorical run registry, gates, and publication style
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import arviz as az
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import xarray as xr
from IPython.display import Markdown, display
from PIL import Image
from scipy.stats import beta as beta_dist, norm

warnings.filterwarnings("default")
warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
    module=r"jupyter_client\..*",
)
pd.set_option("display.max_columns", 180)

TESTED_PYMC_VERSION = "5.28.5"
TESTED_ARVIZ_VERSION = "0.23.4"
TESTED_NUTPIE_VERSION = "0.16.11"
if pm.__version__ != TESTED_PYMC_VERSION:
    warnings.warn(
        f"Validated against PyMC {TESTED_PYMC_VERSION}; runtime has {pm.__version__}."
    )
if az.__version__ != TESTED_ARVIZ_VERSION:
    warnings.warn(
        f"Validated against ArviZ {TESTED_ARVIZ_VERSION}; runtime has {az.__version__}."
    )
runtime_nutpie_version = importlib.metadata.version("nutpie")
if runtime_nutpie_version != TESTED_NUTPIE_VERSION:
    warnings.warn(
        f"Validated against nutpie {TESTED_NUTPIE_VERSION}; runtime has "
        f"{runtime_nutpie_version}."
    )

ANALYSIS_VERSION = (
    "normalized-state-bhm-v2.2c-categorical-separate-compartments-"
    "global-patient-fixed-loading-celltype-nu"
)
MODEL_ARCHITECTURE = "v2p2c_cat_g1_fixed_ctnu"
GATE_VERSION = "publication-only-v1"
PPC_VERSION = "categorical-residual-symmetric-v1"
REVEAL_VERSION = "categorical-rope-bfdr-mc-precision-v1"
NOTEBOOK_RELEASE_SHA256 = "c37d36a0c5a43ff1c8849e64d0efefb4520932c55015ef168b7dd961df95de5b"

BASE = Path("/content/drive/MyDrive/new_dysfunction")
AUDIT_ROOT = BASE / "bayesprism_initial_issue_verification_v1"
PCA_ROOT = AUDIT_ROOT / "publication_pca_tiered_qc_v1"
MODEL_ROOT = BASE / "normalized_state_bhm_v2p2c_categorical_celltype_nu"

PCA_DIR_OVERRIDE = "/content/drive/MyDrive/new_dysfunction/bayesprism_initial_issue_verification_v1/publication_pca_tiered_qc_v1/20260906T013040524557Z"
NOTEBOOK_PATH_OVERRIDE = "/content/drive/MyDrive/new_dysfunction/BHM_normalized_state_categorical_v2p2c_publication.ipynb"
RUN_SPEC = "immune_coarse_primary"  # immune_coarse_primary | immune_fine_secondary | nonimmune_primary
PROFILE = "publication"             # publication | publication_long
WORKFLOW_STAGE = "ppc"              # fit | diagnostics | ppc | ppc_review | reveal | visualize | aggregate
RUN_TRIAGE = False
PPC_JOINT_DRAWS_OVERRIDE = None
PPC_OVERRIDE_APPROVED = False
PPC_OVERRIDE_JUSTIFICATION = ""
REVEAL_RESULTS_APPROVED = False
VISUALIZATION_APPROVED = False

VALID_WORKFLOW_STAGES = {
    "fit", "diagnostics", "ppc", "ppc_review", "reveal", "visualize", "aggregate"
}
if WORKFLOW_STAGE not in VALID_WORKFLOW_STAGES:
    raise ValueError(f"WORKFLOW_STAGE must be one of {sorted(VALID_WORKFLOW_STAGES)}")

RUN_MODEL = WORKFLOW_STAGE == "fit"
RUN_PPC = WORKFLOW_STAGE == "ppc"
RUN_PPC_REVIEW = WORKFLOW_STAGE == "ppc_review"
RUN_DIAGNOSTIC_FIGURES = WORKFLOW_STAGE == "diagnostics"
RUN_RESULT_REVEAL = WORKFLOW_STAGE == "reveal"
RUN_BIOLOGICAL_FIGURES = WORKFLOW_STAGE == "visualize"
RUN_PRIMARY_AGGREGATION = WORKFLOW_STAGE == "aggregate"
LOAD_TRACE_STAGES = {"diagnostics", "ppc", "ppc_review", "reveal", "visualize"}

LIKELIHOOD = "student_t"
NUTS_BACKEND = "nutpie"
BASE_SEED = 20260909

SCORE_VARIANT = "within_component_logcpm_z_cap5"
CLUSTER_REFERENCE_VARIANT = SCORE_VARIANT
FEATURE_MODE = "all"
EXPOSURE_MODEL = "who3group"
BMI_GROUP_LEVELS = ("normal_weight", "overweight", "obese")
FITTED_CONTRASTS = ("overweight_vs_normal", "obese_vs_normal")
REPORTED_CONTRASTS = (
    "overweight_vs_normal", "obese_vs_normal", "obese_vs_overweight"
)
FEATURE_EFFECT_VARIABLES = {
    "overweight_vs_normal": "feature_overweight_vs_normal",
    "obese_vs_normal": "feature_obese_vs_normal",
    "obese_vs_overweight": "feature_obese_vs_overweight",
}
CELLTYPE_EFFECT_VARIABLES = {
    "overweight_vs_normal": "celltype_overweight_vs_normal",
    "obese_vs_normal": "celltype_obese_vs_normal",
    "obese_vs_overweight": "celltype_obese_vs_overweight",
}

IMMUNE_COARSE_TIERED = (("immune_course", "tiered_primary_keep"),)
IMMUNE_FINE_TIERED = (("immune_fine", "tiered_primary_keep"),)
NONIMMUNE_STRICT = (("non_immune", "strict_qc_keep"),)
RUN_SPECS = {
    "immune_coarse_primary": {
        "role": "primary", "analysis_family": "immune_coarse",
        "components": IMMUNE_COARSE_TIERED,
        "expected_celltypes": 8, "expected_features": 306,
    },
    "immune_fine_secondary": {
        "role": "secondary", "analysis_family": "immune_fine",
        "components": IMMUNE_FINE_TIERED,
        "expected_celltypes": 14, "expected_features": 496,
    },
    "nonimmune_primary": {
        "role": "primary", "analysis_family": "nonimmune",
        "components": NONIMMUNE_STRICT,
        "expected_celltypes": 4, "expected_features": 112,
    },
}

PROFILES = {
    "publication": {
        "tune": 2000, "draws": 4000, "chains": 4, "target_accept": 0.99
    },
    "publication_long": {
        "tune": 2000, "draws": 8000, "chains": 4, "target_accept": 0.99
    },
}
PROFILE_PREDECESSOR = {"publication": None, "publication_long": "publication"}

if RUN_SPEC not in RUN_SPECS:
    raise ValueError(f"Unknown RUN_SPEC {RUN_SPEC!r}; choose from {sorted(RUN_SPECS)}")
if PROFILE not in PROFILES:
    raise ValueError(f"Unknown PROFILE {PROFILE!r}; choose publication or publication_long")
ACTIVE_SPEC = RUN_SPECS[RUN_SPEC]

# Priors are on the feature-standardized outcome scale and are identical across
# compartments. They are not tuned using observed categorical results.
BASELINE_SD = 0.50
CELLTYPE_BMI_SD = 0.20
FEATURE_BMI_SD = 0.30
CELLTYPE_COVARIATE_SD = 0.25
FEATURE_COVARIATE_SD = 0.25
PATIENT_GLOBAL_SD = 0.50
LOG_SIGMA_LOCATION = math.log(0.65)
LOG_SIGMA_LOCATION_SD = 0.40
CELLTYPE_LOG_SIGMA_SD = 0.25
FEATURE_LOG_SIGMA_SD = 0.50
NU_MINUS_TWO_RATE = 0.10

ROPE_GRID = (0.05, 0.10, 0.15, 0.20)
PRIMARY_ROPE = 0.10
Q_LEVEL = 0.05

MAX_TREE_DEPTH = 10
MAX_TREE_DEPTH_FRACTION = 0.01
BFMI_MIN = 0.30
FINAL_RHAT_MAX = 1.01
FINAL_ESS_MIN = 400

PPC_SETTINGS = {
    "publication": {"joint_draws": 4000},
    "publication_long": {"joint_draws": 8000},
}
PPC_CORRELATION_THRESHOLD = 0.90

ROPE_INDICATOR_ESS_MIN = 400.0
ROPE_LOCAL_NULL_MCSE_MAX = 0.005
BFDR_MCSE_MAX = 0.0025
MC_ALPHA = 0.025

def stable_run_seed(spec_name: str) -> int:
    offset = int(hashlib.sha256(spec_name.encode()).hexdigest()[:8], 16) % 1_000_000
    return BASE_SEED + offset

RUN_SEED = stable_run_seed(RUN_SPEC)

def make_run_suffix(spec_name: str = RUN_SPEC, profile: str = PROFILE) -> str:
    return (
        f"{MODEL_ARCHITECTURE}__{spec_name}__{SCORE_VARIANT}__"
        f"{FEATURE_MODE}__{EXPOSURE_MODEL}__{profile}"
    )

def run_directory_for(spec_name: str = RUN_SPEC, profile: str = PROFILE) -> Path:
    return MODEL_ROOT / make_run_suffix(spec_name, profile)

OUTPUT_DIR = run_directory_for()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_STYLE = "publication_large_type_v1"
FIGURE_DPI = 600
FIGURE_COLORS = {
    "overweight_vs_normal": "#D95F8D",
    "obese_vs_normal": "#B2182B",
    "obese_vs_overweight": "#6A3D9A",
    "positive": "#D62728", "negative": "#2166AC",
    "neutral": "#666666",
}
HEATMAP_CMAP = "RdBu_r"
FIGURE_ROOT = OUTPUT_DIR / "figures"
FIGURE_DIRECTORIES = {
    category: {
        "tiff": FIGURE_ROOT / category / "tiff",
        "svg": FIGURE_ROOT / category / "svg",
    }
    for category in ("diagnostics", "results")
}
for formats in FIGURE_DIRECTORIES.values():
    for directory in formats.values():
        directory.mkdir(parents=True, exist_ok=True)

def apply_publication_style() -> None:
    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#222222",
        "axes.linewidth": 1.6,
        "axes.titleweight": "bold",
        "axes.titlesize": 20,
        "axes.labelsize": 17,
        "axes.labelweight": "bold",
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 13,
        "legend.title_fontsize": 13,
        "lines.linewidth": 2.2,
        "grid.color": "#B8B8B8",
        "grid.linestyle": "-",
        "grid.linewidth": 0.7,
        "grid.alpha": 0.35,
        "savefig.facecolor": "white",
        "savefig.edgecolor": "none",
        "svg.fonttype": "none",
    })

def save_publication_figure(fig, stem: str, category: str) -> None:
    if category not in FIGURE_DIRECTORIES:
        raise ValueError(f"Unknown figure category: {category}")
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("_")
    tiff_path = FIGURE_DIRECTORIES[category]["tiff"] / f"{safe}.tiff"
    svg_path = FIGURE_DIRECTORIES[category]["svg"] / f"{safe}.svg"
    temporary = tiff_path.with_name(f".{safe}.rendering.tiff")
    finalizing = tiff_path.with_name(f".{safe}.finalizing.tiff")
    for path in (temporary, finalizing):
        if path.exists():
            path.unlink()
    fig.savefig(svg_path, format="svg", bbox_inches="tight")
    compression = "tiff_lzw"
    try:
        fig.savefig(
            temporary, dpi=FIGURE_DPI, format="tiff", bbox_inches="tight",
            pil_kwargs={"compression": compression},
        )
        with Image.open(temporary) as image:
            image.verify()
    except Exception as error:
        warnings.warn(f"LZW TIFF validation failed ({error}); using uncompressed TIFF")
        if temporary.exists():
            temporary.unlink()
        compression = "raw"
        fig.savefig(temporary, dpi=FIGURE_DPI, format="tiff", bbox_inches="tight")
        with Image.open(temporary) as image:
            image.verify()
    shutil.copyfile(temporary, finalizing)
    with Image.open(finalizing) as image:
        image.verify()
    finalizing.replace(tiff_path)
    temporary.unlink()
    print("Saved:", tiff_path, f"({compression})")
    print("Saved:", svg_path)

def cleanup_figure_rendering_temps() -> None:
    gc.collect()
    temporary_paths = (
        list(FIGURE_ROOT.rglob(".*.rendering.tiff"))
        + list(FIGURE_ROOT.rglob(".*.finalizing.tiff"))
    )
    for path in temporary_paths:
        try:
            path.unlink()
        except FileNotFoundError:
            pass

apply_publication_style()
print("Python", platform.python_version(), "| PyMC", pm.__version__, "| ArviZ", az.__version__)
print("Run:", RUN_SPEC, "| profile:", PROFILE, "| stage:", WORKFLOW_STAGE)
print("Architecture:", MODEL_ARCHITECTURE, "| seed:", RUN_SEED)
print("Output:", OUTPUT_DIR)


In [ ]:
# 4. Resolve audited inputs and write the categorical input-audit manifest
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_json(value) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(payload).hexdigest()

def atomic_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    temporary.replace(path)

def atomic_text(value: str, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(value, encoding="utf-8")
    temporary.replace(path)

def atomic_json(value, path: Path) -> None:
    atomic_text(json.dumps(value, indent=2, sort_keys=True, default=str), path)

def strict_bool_series(values: pd.Series, label: str) -> pd.Series:
    true_values = {True, 1, "1", "true", "t", "yes", "y"}
    false_values = {False, 0, "0", "false", "f", "no", "n"}
    parsed = []
    for value in values:
        normalized = value.strip().lower() if isinstance(value, str) else value
        if normalized in true_values:
            parsed.append(True)
        elif normalized in false_values:
            parsed.append(False)
        else:
            raise ValueError(f"{label}: unrecognized Boolean value {value!r}")
    return pd.Series(parsed, index=values.index, dtype=bool)

MUTABLE_NOTEBOOK_ASSIGNMENTS = {
    "PCA_DIR_OVERRIDE", "NOTEBOOK_PATH_OVERRIDE", "RUN_SPEC", "PROFILE",
    "WORKFLOW_STAGE", "RUN_TRIAGE", "PPC_JOINT_DRAWS_OVERRIDE",
    "PPC_OVERRIDE_APPROVED", "PPC_OVERRIDE_JUSTIFICATION",
    "REVEAL_RESULTS_APPROVED", "VISUALIZATION_APPROVED",
    "NOTEBOOK_RELEASE_SHA256",
}

def notebook_code_sha256(path: Path) -> str:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    normalized_sources = []
    assignment = re.compile(
        r"^(" + "|".join(sorted(MUTABLE_NOTEBOOK_ASSIGNMENTS)) + r")\s*=.*$"
    )
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        lines = []
        for line in "".join(cell.get("source", [])).splitlines():
            match = assignment.match(line)
            lines.append(f"{match.group(1)} = <MUTABLE>" if match else line.rstrip())
        normalized_sources.append("\n".join(lines).rstrip())
    return hashlib.sha256("\n\n# CELL\n\n".join(normalized_sources).encode()).hexdigest()

def verify_notebook_release() -> dict:
    if NOTEBOOK_PATH_OVERRIDE is None:
        raise ValueError(
            "Set NOTEBOOK_PATH_OVERRIDE to this saved categorical notebook before running."
        )
    path = Path(NOTEBOOK_PATH_OVERRIDE)
    if not path.exists():
        raise FileNotFoundError(
            f"Saved notebook not found at {path}. Save this notebook there or update "
            "NOTEBOOK_PATH_OVERRIDE."
        )
    actual = notebook_code_sha256(path)
    if actual != NOTEBOOK_RELEASE_SHA256:
        raise ValueError(
            "Notebook code differs from the validated categorical release. Only the "
            "declared user-editable switches may be changed."
        )
    return {"status": "verified", "path": str(path), "sha256": actual}

NOTEBOOK_RELEASE = verify_notebook_release()

def latest_complete_directory(parent: Path, required) -> Path:
    if not parent.exists():
        raise FileNotFoundError(parent)
    candidates = sorted(
        (path for path in parent.iterdir() if path.is_dir()),
        key=lambda path: path.name,
        reverse=True,
    )
    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f"No complete run found under {parent}")

if RUN_PRIMARY_AGGREGATION:
    print('Aggregation stage: current-run data resolution is intentionally skipped.')
else:
    PCA_REQUIRED_FILES = (
        "pca_manifest.json",
        "component_inclusion_tiers.csv",
        "pca_tiered_vs_strict_stability.csv",
        "pca_input_hashes.csv",
    )
    if PCA_DIR_OVERRIDE is None:
        if PROFILE in {"publication", "publication_long"} or WORKFLOW_STAGE in {"reveal", "visualize"}:
            raise ValueError("Publication/reveal requires an exact PCA_DIR_OVERRIDE")
        PCA_DIR = latest_complete_directory(PCA_ROOT, required=PCA_REQUIRED_FILES)
    else:
        PCA_DIR = Path(PCA_DIR_OVERRIDE)
        missing_pca_files = [
            name for name in PCA_REQUIRED_FILES if not (PCA_DIR / name).exists()
        ]
        if missing_pca_files:
            raise FileNotFoundError(
                f"PCA_DIR_OVERRIDE is incomplete: {missing_pca_files}"
            )
    pca_manifest_path = PCA_DIR / "pca_manifest.json"
    pca_manifest = json.loads(pca_manifest_path.read_text(encoding="utf-8"))
    REVIEW_DIR = Path(pca_manifest["combined_review_directory"])
    combined_manifest_path = REVIEW_DIR / "combined_review_manifest.json"
    combined_manifest = json.loads(combined_manifest_path.read_text(encoding="utf-8"))
    RUN_DIRS = {
        compartment: Path(path)
        for compartment, path in combined_manifest["selected_runs"].items()
    }

    tiers_path = PCA_DIR / "component_inclusion_tiers.csv"
    pca_stability_path = PCA_DIR / "pca_tiered_vs_strict_stability.csv"
    cluster_membership_path = REVIEW_DIR / "normalized_correlation_cluster_membership.csv"
    pca_hashes_path = PCA_DIR / "pca_input_hashes.csv"
    tiers = pd.read_csv(tiers_path)
    pca_stability = pd.read_csv(pca_stability_path)
    cluster_membership = pd.read_csv(cluster_membership_path)
    pca_hashes = pd.read_csv(pca_hashes_path)

    # Verify that the audit score files have not changed since PCA.
    hash_mismatches = []
    for row in pca_hashes.itertuples(index=False):
        source = Path(row.path)
        if not source.exists():
            hash_mismatches.append((str(source), "missing"))
        elif sha256_file(source) != row.sha256:
            hash_mismatches.append((str(source), "sha256_changed"))
    if hash_mismatches:
        raise ValueError(f"PCA input provenance changed: {hash_mismatches}")

    def choose_cluster_representatives(
        compartment: str,
        run_dir: Path,
        candidate_features: set[str],
    ) -> set[str]:
        clusters = cluster_membership[
            cluster_membership["compartment"].eq(compartment)
            & cluster_membership["variant"].eq(CLUSTER_REFERENCE_VARIANT)
            & cluster_membership["correlation_kind"].eq(
                "residual_bmi_age_sex_theta"
            )
            & cluster_membership["feature"].isin(candidate_features)
        ][["feature", "cluster_id"]].drop_duplicates()
        if clusters["feature"].duplicated().any():
            raise ValueError(f"{compartment}: a feature belongs to multiple clusters")

        missing_cluster = sorted(candidate_features - set(clusters["feature"]))
        if missing_cluster:
            clusters = pd.concat(
                [
                    clusters,
                    pd.DataFrame(
                        {
                            "feature": missing_cluster,
                            "cluster_id": [
                                f"{compartment}__singleton__{feature}"
                                for feature in missing_cluster
                            ],
                        }
                    ),
                ],
                ignore_index=True,
            )

        metadata_path = run_dir / "score_metadata_all_variants.csv"
        metadata = pd.read_csv(metadata_path)
        metadata = metadata[
            metadata["variant"].eq(CLUSTER_REFERENCE_VARIANT)
            & strict_bool_series(metadata["included"], "score_metadata.included")
        ][["feature", "n_genes_available", "coverage"]].drop_duplicates()
        if metadata["feature"].duplicated().any():
            raise ValueError(f"{compartment}: ambiguous score metadata")

        ranked = clusters.merge(metadata, on="feature", how="left", validate="one_to_one")
        ranked["n_genes_available"] = pd.to_numeric(
            ranked["n_genes_available"], errors="coerce"
        ).fillna(-1)
        ranked["coverage"] = pd.to_numeric(ranked["coverage"], errors="coerce").fillna(-1)
        ranked = ranked.sort_values(
            ["cluster_id", "n_genes_available", "coverage", "feature"],
            ascending=[True, False, False, True],
        )
        representatives = ranked.drop_duplicates("cluster_id", keep="first")
        return set(representatives["feature"])

    cohort_reference = None
    analysis_parts = []
    component_inventory_parts = []
    source_hash_rows = []
    source_feature_count_before_mode = 0

    for compartment, tier_column in RUN_SPECS[RUN_SPEC]["components"]:
        run_dir = RUN_DIRS[compartment]
        cohort_path = run_dir / "analysis_cohort.csv"
        score_path = run_dir / "audit_scores_all_variants.csv.gz"
        patient_qc_path = run_dir / "patient_qc.csv"

        cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
        scores = pd.read_csv(score_path, dtype={"base_sample_id": str})
        patient_qc = pd.read_csv(patient_qc_path, dtype={"base_sample_id": str})

        required_cohort = {
            "base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"
        }
        required_scores = {
            "base_sample_id", "CellType", "Signature", "feature", "variant", "score"
        }
        if not required_cohort.issubset(cohort.columns):
            raise ValueError(f"{compartment}: incomplete cohort table")
        if not required_scores.issubset(scores.columns):
            raise ValueError(f"{compartment}: incomplete audit score table")
        if cohort["base_sample_id"].nunique() != 127 or cohort["base_sample_id"].duplicated().any():
            raise ValueError(f"{compartment}: expected 127 unique patients")
        expected_groups = {"normal_weight": 52, "overweight": 57, "obese": 18}
        if cohort["bmi_group"].value_counts().to_dict() != expected_groups:
            raise ValueError(f"{compartment}: BMI cohort counts changed")

        comparison_columns = [
            "base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"
        ]
        comparison = cohort[comparison_columns].sort_values("base_sample_id").reset_index(drop=True)
        if cohort_reference is None:
            cohort_reference = comparison
        else:
            if not cohort_reference[["base_sample_id", "bmi_group"]].equals(
                comparison[["base_sample_id", "bmi_group"]]
            ):
                raise ValueError("Patient IDs or BMI groups differ across compartments")
            for column in ("bmi", "bmi_z", "age_z", "sex_centered"):
                if not np.allclose(
                    cohort_reference[column], comparison[column], rtol=0, atol=1e-12
                ):
                    raise ValueError(f"Patient covariate {column} differs across compartments")

        local_tiers = tiers[tiers["compartment"].eq(compartment)].copy()
        if tier_column not in local_tiers.columns:
            raise KeyError(f"Missing tier column {tier_column}")
        keep_celltypes = sorted(
            local_tiers.loc[strict_bool_series(local_tiers[tier_column], tier_column), "CellType"].astype(str)
        )
        component_inventory = local_tiers.copy()
        component_inventory["requested_rule"] = tier_column
        component_inventory["included_in_this_run"] = component_inventory["CellType"].isin(
            keep_celltypes
        )
        component_inventory_parts.append(component_inventory)

        local = scores[
            scores["variant"].eq(SCORE_VARIANT)
            & scores["CellType"].isin(keep_celltypes)
        ].copy()
        if local.empty:
            raise ValueError(f"{compartment}: no scores for selected endpoint/components")
        candidate_features = set(local["feature"].astype(str))
        source_feature_count_before_mode += len(candidate_features)
        if FEATURE_MODE == "cluster_representative":
            representatives = choose_cluster_representatives(
                compartment, run_dir, candidate_features
            )
            local = local[local["feature"].isin(representatives)].copy()

        local.insert(0, "source_compartment", compartment)
        local["global_celltype"] = (
            local["source_compartment"].astype(str) + "||" + local["CellType"].astype(str)
        )
        local["global_feature"] = (
            local["source_compartment"].astype(str) + "||" + local["feature"].astype(str)
        )
        local = local.merge(
            cohort[comparison_columns],
            on="base_sample_id",
            how="inner",
            validate="many_to_one",
        )

        analysis_parts.append(local)
        source_hash_rows.extend(
            [
                {"compartment": compartment, "kind": "cohort", "path": str(cohort_path), "sha256": sha256_file(cohort_path)},
                {"compartment": compartment, "kind": "scores", "path": str(score_path), "sha256": sha256_file(score_path)},
                {"compartment": compartment, "kind": "patient_qc", "path": str(patient_qc_path), "sha256": sha256_file(patient_qc_path)},
            ]
        )

    analysis_data = pd.concat(analysis_parts, ignore_index=True)
    if analysis_data.duplicated(["base_sample_id", "global_feature"]).any():
        raise ValueError("Conflicting duplicate patient/feature rows")

    # Standardize each pathway score once across the complete cohort.
    # Categorical coefficients are adjusted mean differences on this scale;
    # they are not Cohen's d or Hedges' g.
    analysis_data["score"] = pd.to_numeric(analysis_data["score"], errors="coerce")
    feature_mean = analysis_data.groupby("global_feature")["score"].transform("mean")
    feature_sd = analysis_data.groupby("global_feature")["score"].transform("std")
    if feature_sd.isna().any() or (feature_sd <= 0).any():
        raise ValueError("At least one selected feature is constant/nonfinite")
    analysis_data["score_model"] = (analysis_data["score"] - feature_mean) / feature_sd
    analysis_data["is_overweight"] = analysis_data["bmi_group"].eq("overweight").astype(float)
    analysis_data["is_obese"] = analysis_data["bmi_group"].eq("obese").astype(float)
    if not np.allclose(
        analysis_data[["is_overweight", "is_obese"]].sum(axis=1),
        analysis_data["bmi_group"].ne("normal_weight").astype(float),
    ):
        raise AssertionError("BMI indicator construction failed")
    required_numeric = [
        "score_model", "is_overweight", "is_obese", "age_z", "sex_centered"
    ]
    if not np.isfinite(analysis_data[required_numeric].to_numpy(float)).all():
        raise ValueError("Analysis table contains missing/nonfinite model values")

    feature_counts = analysis_data.groupby("global_feature")["base_sample_id"].nunique()
    if not feature_counts.eq(127).all():
        raise ValueError("Every modeled feature must contain all 127 patients")

    n_features = analysis_data["global_feature"].nunique()
    n_celltypes = analysis_data["global_celltype"].nunique()
    n_patients = analysis_data["base_sample_id"].nunique()
    expected = RUN_SPECS[RUN_SPEC]
    if source_feature_count_before_mode != expected["expected_features"]:
        raise ValueError(
            f"Expected {expected['expected_features']} source features before feature-mode "
            f"selection; found {source_feature_count_before_mode}"
        )
    if n_celltypes != expected["expected_celltypes"]:
        raise ValueError(
            f"Expected {expected['expected_celltypes']} celltypes; found {n_celltypes}"
        )
    if FEATURE_MODE == "all" and n_features != expected["expected_features"]:
        raise ValueError(
            f"Expected {expected['expected_features']} modeled features; found {n_features}"
        )
    if len(analysis_data) != n_features * n_patients:
        raise ValueError("Analysis table is not a complete patient-by-feature rectangle")

    analysis_data = analysis_data.sort_values(
        ["global_celltype", "global_feature", "base_sample_id"]
    ).reset_index(drop=True)

    analysis_path = OUTPUT_DIR / "analysis_input_long.csv.gz"
    atomic_csv(
        analysis_data,
        analysis_path,
        compression={"method": "gzip", "compresslevel": 6, "mtime": 0},
    )
    component_inventory = pd.concat(component_inventory_parts, ignore_index=True)
    atomic_csv(component_inventory, OUTPUT_DIR / "component_inventory.csv")
    atomic_csv(pd.DataFrame(source_hash_rows), OUTPUT_DIR / "source_input_hashes.csv")

    feature_inventory = analysis_data[
        [
            "source_compartment", "CellType", "Signature", "feature",
            "global_celltype", "global_feature",
        ]
    ].drop_duplicates()
    cluster_lookup = cluster_membership[
        cluster_membership["variant"].eq(CLUSTER_REFERENCE_VARIANT)
        & cluster_membership["correlation_kind"].eq(
            "residual_bmi_age_sex_theta"
        )
    ][["compartment", "feature", "cluster_id"]].drop_duplicates()
    if cluster_lookup.duplicated(["compartment", "feature"]).any():
        raise ValueError("A feature maps to more than one normalized correlation cluster")
    feature_inventory = feature_inventory.merge(
        cluster_lookup,
        left_on=["source_compartment", "feature"],
        right_on=["compartment", "feature"],
        how="left",
        validate="one_to_one",
    ).drop(columns="compartment")
    if feature_inventory["cluster_id"].isna().any():
        missing_cluster_features = feature_inventory.loc[
            feature_inventory["cluster_id"].isna(), "global_feature"
        ].tolist()
        raise ValueError(
            "Normalized correlation-cluster membership is missing for: "
            f"{missing_cluster_features[:10]}"
        )
    feature_inventory["global_cluster_id"] = (
        feature_inventory["source_compartment"].astype(str)
        + "||"
        + feature_inventory["cluster_id"].astype(str)
    )
    feature_inventory["cluster_size_in_run"] = feature_inventory.groupby(
        "global_cluster_id"
    )["global_feature"].transform("nunique")
    feature_inventory = feature_inventory.sort_values("global_feature")
    feature_inventory_path = OUTPUT_DIR / "feature_inventory.csv"
    atomic_csv(feature_inventory, feature_inventory_path)

    preflight_manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "gate_version": GATE_VERSION,
        "notebook_release": NOTEBOOK_RELEASE,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "run_spec": RUN_SPEC,
        "role": RUN_SPECS[RUN_SPEC]["role"],
        "analysis_family": RUN_SPECS[RUN_SPEC]["analysis_family"],
        "profile": PROFILE,
        "score_variant": SCORE_VARIANT,
        "feature_mode": FEATURE_MODE,
        "exposure_model": EXPOSURE_MODEL,
        "reference_group": "normal_weight",
        "fitted_contrasts": list(FITTED_CONTRASTS),
        "reported_contrasts": list(REPORTED_CONTRASTS),
        "likelihood": LIKELIHOOD,
        "nuts_backend": NUTS_BACKEND,
        "seed": RUN_SEED,
        "pca_directory": str(PCA_DIR),
        "pca_manifest_sha256": sha256_file(pca_manifest_path),
        "combined_review_manifest_sha256": sha256_file(combined_manifest_path),
        "component_inclusion_tiers_sha256": sha256_file(tiers_path),
        "pca_stability_sha256": sha256_file(pca_stability_path),
        "pca_input_hashes_sha256": sha256_file(pca_hashes_path),
        "normalized_cluster_membership_sha256": sha256_file(cluster_membership_path),
        "feature_inventory_sha256": sha256_file(feature_inventory_path),
        "analysis_input_sha256": sha256_file(analysis_path),
        "n_patients": n_patients,
        "n_celltypes": n_celltypes,
        "n_features": n_features,
        "n_source_features_before_feature_mode": source_feature_count_before_mode,
        "n_correlation_clusters": int(
            feature_inventory["global_cluster_id"].nunique()
        ),
        "n_multifeature_correlation_clusters": int(
            feature_inventory.loc[
                feature_inventory["cluster_size_in_run"] > 1,
                "global_cluster_id",
            ].nunique()
        ),
        "n_observations": len(analysis_data),
        "cohort_counts": cohort_reference["bmi_group"].value_counts().to_dict(),
        "estimand": {
            "model_parameters": {
                "overweight_vs_normal": (
                    "adjusted mean difference in feature-standardized normalized "
                    "signature score: overweight minus normal weight"
                ),
                "obese_vs_normal": (
                    "adjusted mean difference in feature-standardized normalized "
                    "signature score: obese minus normal weight"
                ),
                "obese_vs_overweight": (
                    "derived adjusted mean difference: obese minus overweight"
                ),
            },
            "adjusted_for": ["age_z", "sex_centered"],
            "reference_group": "normal_weight",
            "not_cohens_d": True,
            "not_hedges_g": True,
            "hedges_g_reported_separately_as_unadjusted_descriptive": True,
        },
        "included_components": component_inventory.loc[
            component_inventory["included_in_this_run"],
            ["compartment", "CellType", "requested_rule"],
        ].to_dict("records"),
        "priors": {
            "baseline_sd": BASELINE_SD,
            "celltype_group_contrast_sd": CELLTYPE_BMI_SD,
            "feature_group_contrast_sd": FEATURE_BMI_SD,
            "celltype_covariate_sd": CELLTYPE_COVARIATE_SD,
            "feature_covariate_sd": FEATURE_COVARIATE_SD,
            "patient_global_sd": PATIENT_GLOBAL_SD,
            "log_sigma_location": LOG_SIGMA_LOCATION,
            "log_sigma_location_sd": LOG_SIGMA_LOCATION_SD,
            "celltype_log_sigma_sd": CELLTYPE_LOG_SIGMA_SD,
            "feature_log_sigma_sd": FEATURE_LOG_SIGMA_SD,
            "nu_minus_two_distribution": "Exponential",
            "nu_minus_two_rate": NU_MINUS_TWO_RATE,
        },
        "rope_grid": list(ROPE_GRID),
        "primary_rope": PRIMARY_ROPE,
        "q_level": Q_LEVEL,
        "patient_effect_structure": {
            "global_patient_effect": "categorical-fixed-effect-orthogonal residual basis",
            "global_patient_loading": 1.0,
            "patient_by_celltype_effect": False,
            "estimated_patient_loadings": False,
            "patient_global_sigma_interpretation": (
                "scale in the normalized residual basis; its square is the "
                "mean marginal patient-effect variance, not an unrestricted "
                "raw between-patient variance"
            ),
        },
        "student_t_nu_scope": "one_nu_per_celltype_within_separately_fitted_compartment",
        "student_t_sigma_interpretation": (
            "sigma_feature is the Student-t scale parameter; conditional SD is "
            "sigma_feature * sqrt(nu_celltype / (nu_celltype - 2))"
        ),
        "versions": {
            "python": platform.python_version(),
            "pymc": pm.__version__,
            "arviz": az.__version__,
            "matplotlib": mpl.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "nutpie": runtime_nutpie_version,
        },
    }
    atomic_text(
        json.dumps(preflight_manifest, indent=2, default=str),
        OUTPUT_DIR / "preflight_manifest.json",
    )

    print(json.dumps(preflight_manifest, indent=2, default=str))
    print("\nPCA stability rows used for design decisions")
    display(
        pca_stability[
            pca_stability["compartment"].isin(
                [item[0] for item in RUN_SPECS[RUN_SPEC]["components"]]
            )
        ]
    )


In [ ]:
# 5. Build and lock the categorical fixed-loading, cell-type-nu model

if RUN_PRIMARY_AGGREGATION:
    model = None
    model_arrays = None
    MODEL_FINGERPRINT = None
    ANALYSIS_LOCK_SHA256 = None
    print("Aggregation stage: current-run model construction is intentionally skipped.")
else:
    def prepare_model_arrays(frame: pd.DataFrame):
        data = frame.sort_values(
            ["global_celltype", "global_feature", "base_sample_id"]
        ).reset_index(drop=True).copy()
        features = sorted(data["global_feature"].unique())
        celltypes = sorted(data["global_celltype"].unique())
        patients = sorted(data["base_sample_id"].unique())
        feature_index = {name: index for index, name in enumerate(features)}
        celltype_index = {name: index for index, name in enumerate(celltypes)}
        patient_index = {name: index for index, name in enumerate(patients)}
        data["feature_idx"] = data["global_feature"].map(feature_index)
        data["celltype_idx"] = data["global_celltype"].map(celltype_index)
        data["patient_idx"] = data["base_sample_id"].map(patient_index)

        feature_mapping = (
            data[["feature_idx", "celltype_idx"]]
            .drop_duplicates()
            .sort_values("feature_idx")
        )
        if (
            len(feature_mapping) != len(features)
            or feature_mapping["feature_idx"].duplicated().any()
        ):
            raise ValueError("Every feature must map to exactly one cell type")
        feature_to_celltype = feature_mapping["celltype_idx"].to_numpy(int)

        patient_predictors = ["is_overweight", "is_obese", "age_z", "sex_centered"]
        consistency = data.groupby("patient_idx")[patient_predictors].nunique(dropna=False)
        if (consistency > 1).to_numpy().any():
            raise ValueError("A patient-level predictor varies across that patient's features")
        patient_table = (
            data[["patient_idx", *patient_predictors]]
            .drop_duplicates("patient_idx")
            .sort_values("patient_idx")
            .reset_index(drop=True)
        )
        if not np.array_equal(patient_table["patient_idx"], np.arange(len(patients))):
            raise ValueError("Patient ordering is inconsistent")

        # The global patient effect is restricted to the residual subspace of
        # intercept + both BMI indicators + age + sex. It therefore cannot absorb
        # any categorical BMI contrast.
        patient_design_columns = ["intercept", *patient_predictors]
        patient_design = np.column_stack([
            np.ones(len(patients)),
            patient_table[patient_predictors].to_numpy(float),
        ])
        if not np.isfinite(patient_design).all():
            raise ValueError("Patient fixed-effect design is nonfinite")
        u, singular_values, _ = np.linalg.svd(patient_design, full_matrices=True)
        tolerance = (
            max(patient_design.shape)
            * np.finfo(float).eps
            * singular_values.max()
        )
        design_rank = int((singular_values > tolerance).sum())
        if design_rank != patient_design.shape[1]:
            raise ValueError("Categorical patient fixed-effect design is rank deficient")
        if design_rank >= len(patients):
            raise ValueError("Patient design leaves no residual random-effect dimension")
        patient_basis = u[:, design_rank:]
        patient_basis *= np.sqrt(len(patients) / patient_basis.shape[1])
        orthogonality_error = float(np.abs(patient_design.T @ patient_basis).max())
        if orthogonality_error > 1e-10:
            raise AssertionError(f"Patient basis is not orthogonal: {orthogonality_error}")

        feature_count_by_celltype = np.bincount(
            feature_to_celltype, minlength=len(celltypes)
        ).astype(int)
        return {
            "data": data,
            "features": features,
            "celltypes": celltypes,
            "patients": patients,
            "feature_to_celltype": feature_to_celltype,
            "patient_basis": patient_basis,
            "patient_design_rank": design_rank,
            "patient_design_column_count": patient_design.shape[1],
            "patient_design_columns": patient_design_columns,
            "patient_orthogonality_error": orthogonality_error,
            "feature_count_by_celltype": feature_count_by_celltype,
        }

    def build_normalized_state_categorical_model(frame: pd.DataFrame):
        arrays = prepare_model_arrays(frame)
        data = arrays["data"]
        features = arrays["features"]
        celltypes = arrays["celltypes"]
        patients = arrays["patients"]
        covariates = ["age_z", "sex_centered"]
        coords = {
            "obs_id": np.arange(len(data)),
            "feature": features,
            "celltype": celltypes,
            "patient": patients,
            "patient_re_dimension": np.arange(arrays["patient_basis"].shape[1]),
            "covariate": covariates,
            "bmi_contrast": list(FITTED_CONTRASTS),
        }

        with pm.Model(coords=coords) as model:
            feature_idx = pm.Data(
                "feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id"
            )
            patient_idx = pm.Data(
                "patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id"
            )
            celltype_idx = pm.Data(
                "celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id"
            )
            feature_to_celltype = pm.Data(
                "feature_to_celltype", arrays["feature_to_celltype"], dims="feature"
            )
            group_indicator_matrix = pm.Data(
                "group_indicator_matrix",
                data[["is_overweight", "is_obese"]].to_numpy(float),
                dims=("obs_id", "bmi_contrast"),
            )
            covariate_matrix = pm.Data(
                "covariate_matrix", data[covariates].to_numpy(float),
                dims=("obs_id", "covariate"),
            )
            y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
            patient_residual_basis = pm.Data(
                "patient_residual_basis", arrays["patient_basis"],
                dims=("patient", "patient_re_dimension"),
            )

            feature_baseline = pm.Normal(
                "feature_baseline", 0.0, BASELINE_SD, dims="feature"
            )

            # Two group contrasts share one predeclared hierarchy. Normal weight
            # is the reference; obese-vs-overweight is derived from the joint
            # posterior and therefore retains the correct covariance.
            celltype_bmi_contrast = pm.Normal(
                "celltype_bmi_contrast", 0.0, CELLTYPE_BMI_SD,
                dims=("bmi_contrast", "celltype"),
            )
            feature_sigma_bmi_contrast = pm.HalfNormal(
                "feature_sigma_bmi_contrast", FEATURE_BMI_SD,
                dims=("bmi_contrast", "celltype"),
            )
            feature_bmi_raw = pm.Normal(
                "feature_bmi_raw", 0.0, 1.0,
                dims=("bmi_contrast", "feature"),
            )
            feature_bmi_contrast = pm.Deterministic(
                "feature_bmi_contrast",
                celltype_bmi_contrast[:, feature_to_celltype]
                + feature_bmi_raw
                * feature_sigma_bmi_contrast[:, feature_to_celltype],
                dims=("bmi_contrast", "feature"),
            )
            pm.Deterministic(
                "feature_overweight_vs_normal", feature_bmi_contrast[0], dims="feature"
            )
            pm.Deterministic(
                "feature_obese_vs_normal", feature_bmi_contrast[1], dims="feature"
            )
            pm.Deterministic(
                "feature_obese_vs_overweight",
                feature_bmi_contrast[1] - feature_bmi_contrast[0],
                dims="feature",
            )
            pm.Deterministic(
                "celltype_overweight_vs_normal", celltype_bmi_contrast[0],
                dims="celltype",
            )
            pm.Deterministic(
                "celltype_obese_vs_normal", celltype_bmi_contrast[1],
                dims="celltype",
            )
            pm.Deterministic(
                "celltype_obese_vs_overweight",
                celltype_bmi_contrast[1] - celltype_bmi_contrast[0],
                dims="celltype",
            )

            celltype_covariate_slope = pm.Normal(
                "celltype_covariate_slope", 0.0, CELLTYPE_COVARIATE_SD,
                dims=("covariate", "celltype"),
            )
            feature_sigma_covariate = pm.HalfNormal(
                "feature_sigma_covariate", FEATURE_COVARIATE_SD,
                dims=("covariate", "celltype"),
            )
            feature_covariate_raw = pm.Normal(
                "feature_covariate_raw", 0.0, 1.0,
                dims=("covariate", "feature"),
            )
            feature_covariate_slope = pm.Deterministic(
                "feature_covariate_slope",
                celltype_covariate_slope[:, feature_to_celltype]
                + feature_covariate_raw
                * feature_sigma_covariate[:, feature_to_celltype],
                dims=("covariate", "feature"),
            )

            patient_global_sigma = pm.HalfNormal(
                "patient_global_sigma", PATIENT_GLOBAL_SD
            )
            patient_global_coefficient = pm.Normal(
                "patient_global_coefficient", 0.0, patient_global_sigma,
                dims="patient_re_dimension",
            )
            patient_global_effect = pm.Deterministic(
                "patient_global_effect",
                pt.dot(patient_residual_basis, patient_global_coefficient),
                dims="patient",
            )

            contrast_at_observation = feature_bmi_contrast[:, feature_idx].T
            covariate_at_observation = feature_covariate_slope[:, feature_idx].T
            mu = (
                feature_baseline[feature_idx]
                + pt.sum(group_indicator_matrix * contrast_at_observation, axis=1)
                + pt.sum(covariate_matrix * covariate_at_observation, axis=1)
                + patient_global_effect[patient_idx]
            )

            log_sigma_location = pm.Normal(
                "log_sigma_location", LOG_SIGMA_LOCATION, LOG_SIGMA_LOCATION_SD
            )
            log_sigma_celltype_location_sd = pm.HalfNormal(
                "log_sigma_celltype_location_sd", CELLTYPE_LOG_SIGMA_SD
            )
            log_sigma_celltype_raw = pm.Normal(
                "log_sigma_celltype_raw", 0.0, 1.0, dims="celltype"
            )
            log_sigma_celltype_location = pm.Deterministic(
                "log_sigma_celltype_location",
                log_sigma_location
                + log_sigma_celltype_location_sd * log_sigma_celltype_raw,
                dims="celltype",
            )
            log_sigma_feature_scale = pm.HalfNormal(
                "log_sigma_feature_scale", FEATURE_LOG_SIGMA_SD, dims="celltype"
            )
            log_sigma_raw = pm.Normal(
                "log_sigma_raw", 0.0, 1.0, dims="feature"
            )
            sigma_feature = pm.Deterministic(
                "sigma_feature",
                pt.exp(
                    log_sigma_celltype_location[feature_to_celltype]
                    + log_sigma_feature_scale[feature_to_celltype] * log_sigma_raw
                ),
                dims="feature",
            )
            nu_minus_two = pm.Exponential(
                "nu_minus_two", lam=NU_MINUS_TWO_RATE, dims="celltype"
            )
            nu = pm.Deterministic("nu", nu_minus_two + 2.0, dims="celltype")
            pm.StudentT(
                "y_obs", nu=nu[celltype_idx], mu=mu,
                sigma=sigma_feature[feature_idx], observed=y, dims="obs_id",
            )
        return model, arrays

    model, model_arrays = build_normalized_state_categorical_model(analysis_data)
    initial_logp = float(model.compile_logp()(model.initial_point()))
    if not np.isfinite(initial_logp):
        raise ValueError(f"Initial model log-probability is not finite: {initial_logp}")

    expected_free_rvs = {
        "feature_baseline", "celltype_bmi_contrast",
        "feature_sigma_bmi_contrast", "feature_bmi_raw",
        "celltype_covariate_slope", "feature_sigma_covariate",
        "feature_covariate_raw", "patient_global_sigma",
        "patient_global_coefficient", "log_sigma_location",
        "log_sigma_celltype_location_sd", "log_sigma_celltype_raw",
        "log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two",
    }
    actual_free_rvs = {variable.name for variable in model.free_RVs}
    if actual_free_rvs != expected_free_rvs:
        raise AssertionError(
            f"Free-RV contract changed; missing={sorted(expected_free_rvs-actual_free_rvs)}, "
            f"extra={sorted(actual_free_rvs-expected_free_rvs)}"
        )
    forbidden_names = {
        "bmi_z", "feature_bmi_slope", "celltype_bmi_slope",
        "patient_celltype_sigma", "patient_celltype_raw", "patient_celltype_effect",
        "global_loading", "celltype_loading", "patient_global_raw",
    }
    present_forbidden = sorted(forbidden_names & set(model.named_vars))
    if present_forbidden:
        raise AssertionError(f"Forbidden variables are present: {present_forbidden}")
    if model["nu"].ndim != 1:
        raise AssertionError("The categorical v2.2c model requires one nu per cell type")

    required_derived_diagnostics = [
        "feature_bmi_contrast", "feature_overweight_vs_normal",
        "feature_obese_vs_normal", "feature_obese_vs_overweight",
        "celltype_overweight_vs_normal", "celltype_obese_vs_normal",
        "celltype_obese_vs_overweight", "feature_covariate_slope",
        "patient_global_effect", "log_sigma_celltype_location",
        "sigma_feature", "nu",
    ]
    model_structure = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "initial_logp": initial_logp,
        "n_free_rv_families": len(model.free_RVs),
        "n_unconstrained_scalar_coordinates": int(sum(
            np.size(value) for value in model.initial_point().values()
        )),
        "free_rvs": sorted(actual_free_rvs),
        "required_derived_diagnostics": sorted(required_derived_diagnostics),
        "reference_group": "normal_weight",
        "fitted_contrasts": list(FITTED_CONTRASTS),
        "reported_contrasts": list(REPORTED_CONTRASTS),
        "patient_design_rank": model_arrays["patient_design_rank"],
        "patient_design_column_count": model_arrays["patient_design_column_count"],
        "patient_design_columns": model_arrays["patient_design_columns"],
        "patient_re_dimensions": model_arrays["patient_basis"].shape[1],
        "patient_orthogonality_max_abs_error": model_arrays["patient_orthogonality_error"],
        "patient_global_sigma_interpretation": (
            "normalized residual-basis scale: its square equals the mean marginal "
            "patient-effect variance; individual variances depend on design leverage"
        ),
        "global_patient_loading": 1.0,
        "patient_by_celltype_effect": False,
        "nu_scope": "celltype_within_separate_compartment",
        "sigma_feature_interpretation": (
            "Student-t scale parameter; conditional residual SD equals "
            "sigma_feature * sqrt(nu_celltype / (nu_celltype - 2))"
        ),
        "feature_count_by_celltype": dict(zip(
            model_arrays["celltypes"],
            model_arrays["feature_count_by_celltype"].tolist(),
        )),
    }
    atomic_json(model_structure, OUTPUT_DIR / "model_structure.json")

    model_fingerprint_payload = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "gate_version": GATE_VERSION,
        "ppc_version": PPC_VERSION,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "run_spec": RUN_SPEC,
        "score_variant": SCORE_VARIANT,
        "feature_mode": FEATURE_MODE,
        "exposure_model": EXPOSURE_MODEL,
        "reference_group": "normal_weight",
        "fitted_contrasts": list(FITTED_CONTRASTS),
        "reported_contrasts": list(REPORTED_CONTRASTS),
        "likelihood": LIKELIHOOD,
        "nuts_backend": NUTS_BACKEND,
        "analysis_input_sha256": preflight_manifest["analysis_input_sha256"],
        "pca_manifest_sha256": preflight_manifest["pca_manifest_sha256"],
        "combined_review_manifest_sha256": preflight_manifest["combined_review_manifest_sha256"],
        "component_inclusion_tiers_sha256": preflight_manifest["component_inclusion_tiers_sha256"],
        "pca_stability_sha256": preflight_manifest["pca_stability_sha256"],
        "pca_input_hashes_sha256": preflight_manifest["pca_input_hashes_sha256"],
        "normalized_cluster_membership_sha256": preflight_manifest["normalized_cluster_membership_sha256"],
        "feature_inventory_sha256": preflight_manifest["feature_inventory_sha256"],
        "priors": preflight_manifest["priors"],
        "free_rvs": sorted(actual_free_rvs),
        "required_derived_diagnostics": sorted(required_derived_diagnostics),
    }
    MODEL_FINGERPRINT = sha256_json(model_fingerprint_payload)
    analysis_lock = {
        **model_fingerprint_payload,
        "model_fingerprint": MODEL_FINGERPRINT,
        "profile": PROFILE,
        "sampler_profile": PROFILES[PROFILE],
        "run_seed": RUN_SEED,
        "rope_grid": list(ROPE_GRID),
        "primary_rope": PRIMARY_ROPE,
        "q_level": Q_LEVEL,
    }
    ANALYSIS_LOCK_PATH = OUTPUT_DIR / "analysis_lock.json"
    canonical_lock = json.dumps(analysis_lock, indent=2, sort_keys=True, default=str)
    if ANALYSIS_LOCK_PATH.exists():
        existing_lock = ANALYSIS_LOCK_PATH.read_text(encoding="utf-8")
        if existing_lock != canonical_lock:
            raise RuntimeError(
                "The write-once analysis lock differs from the current code/input/config. "
                "Use a new RUN_SPEC/profile or restore the locked notebook."
            )
    else:
        atomic_text(canonical_lock, ANALYSIS_LOCK_PATH)
    ANALYSIS_LOCK_SHA256 = sha256_file(ANALYSIS_LOCK_PATH)

    print(json.dumps(model_structure, indent=2, default=str))
    print("Model fingerprint:", MODEL_FINGERPRINT)
    print("Analysis-lock SHA256:", ANALYSIS_LOCK_SHA256)


In [ ]:
# 6. Sample/load atomically and run publication convergence gates

def netcdf_safe_attr(value):
    if value is None:
        return "null"
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (str, bytes, int, float)):
        return value
    if isinstance(value, np.ndarray) and value.dtype != object:
        return value
    return json.dumps(value, default=str, sort_keys=True)

def sanitize_idata_attrs(idata):
    owners = [idata]
    for group_name in idata.groups():
        dataset = getattr(idata, group_name)
        owners.append(dataset)
        owners.extend(dataset.variables.values())
    for owner in owners:
        attrs = getattr(owner, "attrs", None)
        if attrs is not None:
            for key, value in list(attrs.items()):
                attrs[key] = netcdf_safe_attr(value)
    return idata

def save_idata_atomic(idata, path: Path) -> Path:
    temporary = path.with_name(f".{path.name}.tmp")
    if temporary.exists():
        temporary.unlink()
    sanitize_idata_attrs(idata)
    try:
        idata.to_netcdf(temporary)
        temporary.replace(path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return path

def require_publication_long_trigger() -> dict | None:
    if PROFILE == "publication":
        return None
    predecessor_dir = run_directory_for(RUN_SPEC, "publication")
    diagnostics_path = predecessor_dir / "run_diagnostics.json"
    trace_path = predecessor_dir / "posterior_trace.nc"
    lock_path = predecessor_dir / "analysis_lock.json"
    required = [diagnostics_path, trace_path, lock_path]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f"publication_long requires the publication record: {missing}")
    previous = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    identity_checks = {
        "profile": previous.get("profile") == "publication",
        "architecture": previous.get("model_architecture") == MODEL_ARCHITECTURE,
        "fingerprint": previous.get("model_fingerprint") == MODEL_FINGERPRINT,
        "input": previous.get("analysis_input_sha256") == preflight_manifest["analysis_input_sha256"],
        "trace": previous.get("posterior_trace_sha256") == sha256_file(trace_path),
        "lock": previous.get("analysis_lock_sha256") == sha256_file(lock_path),
        "release": previous.get("notebook_release_status") == "verified",
    }
    failed_identity = [name for name, passed in identity_checks.items() if not passed]
    if failed_identity:
        raise RuntimeError(f"publication_long predecessor identity failed: {failed_identity}")

    geometry_pass = previous.get("sampler_geometry_pass") is True
    convergence_precision_failure = bool(
        geometry_pass and previous.get("strict_all_parameter_pass") is False
    )
    precision_path = predecessor_dir / "reveal_precision_gate.json"
    bfdr_precision_failure = False
    if precision_path.exists():
        precision = json.loads(precision_path.read_text(encoding="utf-8"))
        bfdr_precision_failure = bool(
            previous.get("current_profile_gate_pass") is True
            and precision.get("selection_status") == "withheld_precision_failed"
            and precision.get("model_fingerprint") == MODEL_FINGERPRINT
            and precision.get("posterior_trace_sha256") == sha256_file(trace_path)
        )
    if not geometry_pass:
        raise RuntimeError(
            "publication had a sampler-geometry failure. More draws are not an "
            "authorized remedy; inspect divergences, BFMI, and tree depth."
        )
    if not (convergence_precision_failure or bfdr_precision_failure):
        raise RuntimeError(
            "publication_long has no prespecified trigger. Do not run it merely "
            "because the publication effects are weak or null."
        )
    return {
        "profile": "publication",
        "directory": str(predecessor_dir),
        "trigger": (
            "strict_rhat_or_ess_precision_failure"
            if convergence_precision_failure
            else "rope_bfdr_mc_precision_failure"
        ),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "trace_sha256": sha256_file(trace_path),
        "analysis_lock_sha256": sha256_file(lock_path),
        **({"reveal_precision_gate_sha256": sha256_file(precision_path)}
           if bfdr_precision_failure else {}),
    }

TRACE_PATH = OUTPUT_DIR / "posterior_trace.nc"
promotion_evidence = None
if RUN_MODEL:
    if TRACE_PATH.exists():
        raise FileExistsError(
            f"{TRACE_PATH} exists. Load it with another stage; never overwrite a trace."
        )
    promotion_evidence = require_publication_long_trigger()
    sampler = PROFILES[PROFILE]
    sample_options = {
        "tune": sampler["tune"],
        "draws": sampler["draws"],
        "chains": sampler["chains"],
        "cores": sampler["chains"],
        "target_accept": sampler["target_accept"],
        "nuts_sampler": NUTS_BACKEND,
        "random_seed": RUN_SEED,
        "return_inferencedata": True,
    }
    if NUTS_BACKEND == "nutpie":
        sample_options["nuts_sampler_kwargs"] = {"maxdepth": MAX_TREE_DEPTH}
    with model:
        idata = pm.sample(**sample_options)
    trace_lock_attrs = {
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "model_architecture": MODEL_ARCHITECTURE,
        "promotion_evidence": json.dumps(promotion_evidence, sort_keys=True),
    }
    for key, value in trace_lock_attrs.items():
        idata.attrs[key] = value
        idata.posterior.attrs[key] = value
    save_idata_atomic(idata, TRACE_PATH)
elif WORKFLOW_STAGE in LOAD_TRACE_STAGES:
    if not TRACE_PATH.exists():
        raise FileNotFoundError(f"No trace exists for this exact run: {TRACE_PATH}")
    idata = az.from_netcdf(TRACE_PATH)
    expected_attrs = {
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "model_architecture": MODEL_ARCHITECTURE,
    }
    observed_attrs = {
        key: idata.attrs.get(key, idata.posterior.attrs.get(key))
        for key in expected_attrs
    }
    failures = {
        key: {"expected": expected, "observed": observed_attrs.get(key)}
        for key, expected in expected_attrs.items()
        if observed_attrs.get(key) != expected
    }
    if failures:
        raise RuntimeError(f"Trace/analysis-lock mismatch: {failures}")
    stored_promotion = idata.attrs.get(
        "promotion_evidence", idata.posterior.attrs.get("promotion_evidence", "null")
    )
    promotion_evidence = json.loads(stored_promotion)
    print("Loaded locked trace:", TRACE_PATH)
else:
    idata = None
    print("No trace loaded for the aggregation stage.")

def parameter_family(parameter: str) -> str:
    return re.sub(r"\[.*$", "", str(parameter))

if idata is not None:
    posterior_names = set(idata.posterior.data_vars)
    free_names = [variable.name for variable in model.free_RVs]
    missing_free = sorted(set(free_names) - posterior_names)
    missing_derived = sorted(set(required_derived_diagnostics) - posterior_names)
    if missing_free:
        raise RuntimeError(f"Trace lacks free variables: {missing_free}")
    diagnostic_names = sorted(set(free_names + required_derived_diagnostics) & posterior_names)
    full_diagnostics = az.summary(
        idata, var_names=diagnostic_names, round_to="none", kind="diagnostics"
    ).reset_index(names="parameter")
    full_diagnostics["parameter_family"] = full_diagnostics["parameter"].map(
        parameter_family
    )
    atomic_csv(full_diagnostics, OUTPUT_DIR / "full_diagnostics.csv")

    diagnostic_values_finite = bool(np.isfinite(
        full_diagnostics[["r_hat", "ess_bulk", "ess_tail"]].to_numpy(float)
    ).all())
    available_sample_stats = (
        set(idata.sample_stats.data_vars) if hasattr(idata, "sample_stats") else set()
    )
    required_sample_stats = {"diverging", "energy", "n_steps"}
    missing_sample_stats = sorted(required_sample_stats - available_sample_stats)
    step_size_stats = sorted(
        available_sample_stats & {"step_size", "step_size_bar", "step_size_nom"}
    )
    sample_stats_complete = bool(not missing_sample_stats and step_size_stats)
    divergence_count = None
    bfmi = np.array([np.nan])
    maxdepth_fraction_by_chain = []
    maxdepth_detection_method = None
    sample_stats_finite = False
    if sample_stats_complete:
        stats_to_check = required_sample_stats | set(step_size_stats)
        arrays_to_check = {
            name: np.asarray(idata.sample_stats[name], dtype=float)
            for name in stats_to_check
        }
        sample_stats_finite = bool(
            all(np.isfinite(values).all() for values in arrays_to_check.values())
            and (arrays_to_check["n_steps"] > 0).all()
            and all((arrays_to_check[name] > 0).all() for name in step_size_stats)
            and np.isin(arrays_to_check["diverging"], [0.0, 1.0]).all()
        )
        divergence_count = int(arrays_to_check["diverging"].sum())
        try:
            bfmi = np.asarray(az.bfmi(idata), dtype=float)
        except Exception:
            bfmi = np.array([np.nan])
        if "maxdepth_reached" in available_sample_stats:
            reached = np.asarray(idata.sample_stats["maxdepth_reached"], dtype=float)
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(reached).all()
                and np.isin(reached, [0.0, 1.0]).all()
            )
            maxdepth_fraction_by_chain = reached.mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "maxdepth_reached"
        elif "reached_max_treedepth" in available_sample_stats:
            reached = np.asarray(
                idata.sample_stats["reached_max_treedepth"], dtype=float
            )
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(reached).all()
                and np.isin(reached, [0.0, 1.0]).all()
            )
            maxdepth_fraction_by_chain = reached.mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "reached_max_treedepth"
        elif "depth" in available_sample_stats:
            depth = np.asarray(idata.sample_stats["depth"], dtype=float)
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(depth).all() and (depth >= 0).all()
            )
            maxdepth_fraction_by_chain = (
                depth >= MAX_TREE_DEPTH
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "depth"
        elif "tree_depth" in available_sample_stats:
            depth = np.asarray(idata.sample_stats["tree_depth"], dtype=float)
            sample_stats_finite = bool(
                sample_stats_finite and np.isfinite(depth).all() and (depth >= 0).all()
            )
            maxdepth_fraction_by_chain = (
                depth >= MAX_TREE_DEPTH
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "tree_depth"
        else:
            max_steps = 2 ** MAX_TREE_DEPTH - 1
            maxdepth_fraction_by_chain = (
                arrays_to_check["n_steps"] >= max_steps
            ).mean(axis=1).astype(float).tolist()
            maxdepth_detection_method = "n_steps_fallback"

    expected_shape = PROFILES[PROFILE]
    trace_shape_pass = bool(
        idata.posterior.sizes.get("chain") == expected_shape["chains"]
        and idata.posterior.sizes.get("draw") == expected_shape["draws"]
    )
    sampler_geometry_pass = bool(
        sample_stats_complete and sample_stats_finite and divergence_count == 0
        and bfmi.size == expected_shape["chains"]
        and np.isfinite(bfmi).all() and np.min(bfmi) >= BFMI_MIN
        and len(maxdepth_fraction_by_chain) == expected_shape["chains"]
        and max(maxdepth_fraction_by_chain) < MAX_TREE_DEPTH_FRACTION
        and trace_shape_pass
    )
    strict_all_parameter_pass = bool(
        diagnostic_values_finite and not missing_derived
        and (full_diagnostics["r_hat"] < FINAL_RHAT_MAX).all()
        and (full_diagnostics["ess_bulk"] >= FINAL_ESS_MIN).all()
        and (full_diagnostics["ess_tail"] >= FINAL_ESS_MIN).all()
    )
    publication_convergence_pass = bool(
        sampler_geometry_pass and strict_all_parameter_pass
    )
    current_profile_gate_pass = publication_convergence_pass

    family_summary = full_diagnostics.groupby(
        "parameter_family", as_index=False
    ).agg(
        n_rows=("parameter", "size"),
        maximum_rhat=("r_hat", "max"),
        minimum_bulk_ess=("ess_bulk", "min"),
        minimum_tail_ess=("ess_tail", "min"),
    ).sort_values(["maximum_rhat", "minimum_bulk_ess"], ascending=[False, True])
    atomic_csv(family_summary, OUTPUT_DIR / "diagnostic_family_summary.csv")
    gate_table = pd.DataFrame([
        {"gate": "trace_shape", "pass": trace_shape_pass},
        {"gate": "sampler_geometry", "pass": sampler_geometry_pass},
        {"gate": "strict_all_parameters", "pass": strict_all_parameter_pass},
        {"gate": "current_profile", "pass": current_profile_gate_pass},
    ])
    atomic_csv(gate_table, OUTPUT_DIR / "diagnostic_gate_table.csv")
    diagnostics_record = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "analysis_input_sha256": preflight_manifest["analysis_input_sha256"],
        "run_spec": RUN_SPEC,
        "role": ACTIVE_SPEC["role"],
        "analysis_family": ACTIVE_SPEC["analysis_family"],
        "profile": PROFILE,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "notebook_release_status": NOTEBOOK_RELEASE.get("status"),
        "notebook_release_sha256": NOTEBOOK_RELEASE.get("sha256"),
        "available_sample_stats": sorted(available_sample_stats),
        "missing_required_sample_stats": missing_sample_stats,
        "available_step_size_stats": step_size_stats,
        "sample_stats_complete": sample_stats_complete,
        "sample_stats_finite": sample_stats_finite,
        "expected_chains": expected_shape["chains"],
        "observed_chains": idata.posterior.sizes.get("chain"),
        "expected_draws_per_chain": expected_shape["draws"],
        "observed_draws_per_chain": idata.posterior.sizes.get("draw"),
        "trace_shape_pass": trace_shape_pass,
        "divergences": divergence_count,
        "bfmi_by_chain": bfmi.tolist(),
        "min_bfmi": float(np.nanmin(bfmi)),
        "maxdepth_fraction_by_chain": maxdepth_fraction_by_chain,
        "maxdepth_detection_method": maxdepth_detection_method,
        "missing_required_derived_diagnostics": missing_derived,
        "max_all_parameter_rhat": float(full_diagnostics["r_hat"].max()),
        "min_all_parameter_ess_bulk": float(full_diagnostics["ess_bulk"].min()),
        "min_all_parameter_ess_tail": float(full_diagnostics["ess_tail"].min()),
        "sampler_geometry_pass": sampler_geometry_pass,
        "strict_all_parameter_pass": strict_all_parameter_pass,
        "publication_convergence_pass": publication_convergence_pass,
        "current_profile_gate_pass": current_profile_gate_pass,
        "selection_status": "withheld_not_computed",
        "selection_revealed": False,
        "selection_reveal_utc": None,
        "promotion_evidence": promotion_evidence,
    }
    atomic_json(diagnostics_record, OUTPUT_DIR / "run_diagnostics.json")
    print(json.dumps(diagnostics_record, indent=2))


In [ ]:
# 6a. Optional failed-fit triage (diagnostic metrics only; no effects)
if RUN_TRIAGE:
    if idata is None:
        raise ValueError("RUN_TRIAGE requires a loaded trace")
    if diagnostics_record["current_profile_gate_pass"]:
        print("The current gate passed; triage is unnecessary.")
    else:
        worst = full_diagnostics.sort_values(
            ["r_hat", "ess_bulk"], ascending=[False, True]
        ).head(100)
        chain_rows = []
        for variable in diagnostic_names:
            if variable not in idata.posterior:
                continue
            values = idata.posterior[variable]
            parameter_dims = [d for d in values.dims if d not in {"chain", "draw"}]
            array = np.asarray(values.transpose("chain", "draw", *parameter_dims), float)
            array = array.reshape(array.shape[0], array.shape[1], -1)
            chain_means = array.mean(axis=1)
            pooled_sd = array.reshape(-1, array.shape[-1]).std(axis=0, ddof=1)
            separation = (chain_means.max(axis=0) - chain_means.min(axis=0)) / np.where(
                pooled_sd > 0, pooled_sd, np.nan
            )
            chain_rows.append({
                "variable": variable,
                "maximum_chain_mean_separation_in_posterior_sd": float(np.nanmax(separation)),
            })
        chain_separation = pd.DataFrame(chain_rows).sort_values(
            "maximum_chain_mean_separation_in_posterior_sd", ascending=False
        )
        geometry_rows = []
        for variable in sorted(available_sample_stats):
            values = np.asarray(idata.sample_stats[variable])
            if not np.issubdtype(values.dtype, np.number):
                continue
            for chain in range(values.shape[0]):
                geometry_rows.append({
                    "variable": variable, "chain": chain,
                    "median": float(np.nanmedian(values[chain])),
                    "maximum": float(np.nanmax(values[chain])),
                })
        atomic_csv(worst, OUTPUT_DIR / "triage_worst_diagnostic_rows.csv")
        atomic_csv(chain_separation, OUTPUT_DIR / "triage_chain_separation.csv")
        atomic_csv(pd.DataFrame(geometry_rows), OUTPUT_DIR / "triage_sampler_geometry.csv")
        display(family_summary, worst, chain_separation)
else:
    print("Optional triage disabled.")


In [ ]:
# 7. Categorical residual-based posterior predictive checks
# Uses residual dispersion, studentized tails, and raw/conditional dependence.

PPC_V2_VERSION = "categorical-residual-symmetric-v1"
PPC_V2_BATCH_MB = 256
PPC_V2_ALPHA = 0.025

def _ppc_v2_finite(name, values):
    values = np.asarray(values)
    if values.size == 0 or not np.issubdtype(values.dtype, np.number):
        raise RuntimeError(f"{name} is empty or nonnumeric")
    if not np.isfinite(values.astype(float, copy=False)).all():
        raise RuntimeError(f"{name} contains nonfinite values")
    return values

def _ppc_v2_atomic_npz(path: Path, **arrays):
    temporary = path.with_name(f".{path.stem}.tmp.npz")
    if temporary.exists():
        temporary.unlink()
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)

def _ppc_v2_posterior_mean(posterior, name, dimensions):
    array = posterior[name].mean(dim=("chain", "draw"))
    return np.asarray(array.transpose(*dimensions), dtype=float)

def _ppc_v2_flat_draws(posterior_batch, name, dimensions):
    array = posterior_batch[name].transpose("chain", "draw", *dimensions)
    values = np.asarray(array, dtype=float)
    return values.reshape((-1,) + values.shape[2:])

def _ppc_v2_residual_statistics(batch_feature_patient):
    values = _ppc_v2_finite("residual matrix", batch_feature_patient).astype(float)
    residual_sd = values.std(axis=2, ddof=1)
    if (residual_sd <= 0).any():
        raise RuntimeError("A residual feature has zero variance")
    centered = values - values.mean(axis=2, keepdims=True)
    studentized_max_abs = np.max(np.abs(centered), axis=2) / residual_sd
    return residual_sd, studentized_max_abs

def _ppc_v2_dependence_count(batch_feature_patient, columns, threshold):
    values = _ppc_v2_finite(
        "dependence matrix", batch_feature_patient[:, columns, :]
    ).astype(float)
    if values.shape[1] < 2:
        return np.zeros(values.shape[0], dtype=np.int32)
    values = values - values.mean(axis=2, keepdims=True)
    norms = np.sqrt(np.sum(values * values, axis=2))
    if (norms <= 0).any():
        raise RuntimeError("A dependence feature has zero variance")
    numerator = np.einsum("bip,bjp->bij", values, values, optimize=True)
    denominator = norms[:, :, None] * norms[:, None, :]
    correlation = numerator / denominator
    upper = np.triu_indices(values.shape[1], k=1)
    return np.sum(
        np.abs(correlation[:, upper[0], upper[1]]) >= threshold,
        axis=1,
    ).astype(np.int32)

def _ppc_v2_upper_p(reference, observed):
    reference = np.asarray(reference, dtype=float).reshape(-1)
    return float((1 + np.sum(reference >= observed)) / (len(reference) + 1))

if RUN_PPC:
    if idata is None:
        raise RuntimeError("Load the saved posterior trace before running PPC v2")

    diagnostics_path = OUTPUT_DIR / "run_diagnostics.json"
    diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    if diagnostics.get("current_profile_gate_pass") is not True:
        raise RuntimeError("PPC v2 requires a converged saved trace")
    if diagnostics.get("posterior_trace_sha256") != sha256_file(TRACE_PATH):
        raise RuntimeError("The loaded trace differs from run_diagnostics.json")

    requested_joint = int(
        PPC_JOINT_DRAWS_OVERRIDE
        if PPC_JOINT_DRAWS_OVERRIDE is not None
        else PPC_SETTINGS[PROFILE]["joint_draws"]
    )
    n_chains = int(idata.posterior.sizes["chain"])
    stored_draws = int(idata.posterior.sizes["draw"])
    if requested_joint % (2 * n_chains):
        raise ValueError("PPC draws must split evenly by chain and into two halves")
    per_chain = requested_joint // n_chains
    if per_chain > stored_draws:
        raise ValueError("The trace has too few posterior draws for this PPC")
    selected_positions = np.linspace(
        0, stored_draws - 1, per_chain
    ).round().astype(int)
    if len(np.unique(selected_positions)) != per_chain:
        raise RuntimeError("Posterior draw-position selection is not unique")

    data = model_arrays["data"]
    observed = data["score_model"].to_numpy(float)
    feature_idx = data["feature_idx"].to_numpy(int)
    patient_idx = data["patient_idx"].to_numpy(int)
    group_indicators = data[["is_overweight", "is_obese"]].to_numpy(float)
    covariates = data[["age_z", "sex_centered"]].to_numpy(float)
    n_observations = len(observed)
    n_features = len(model_arrays["features"])
    n_patients = len(model_arrays["patients"])
    n_celltypes = len(model_arrays["celltypes"])

    observation_grid = np.full((n_features, n_patients), -1, dtype=np.int64)
    pair_code = feature_idx * n_patients + patient_idx
    if np.unique(pair_code).size != n_observations:
        raise ValueError("Duplicate patient-feature observations")
    observation_grid[feature_idx, patient_idx] = np.arange(n_observations)
    if (observation_grid < 0).any():
        raise ValueError("PPC v2 requires a complete patient-feature rectangle")

    feature_columns_by_celltype = {
        j: np.flatnonzero(model_arrays["feature_to_celltype"] == j)
        for j in range(n_celltypes)
    }

    # Posterior-mean fitted value for the fixed observed residual matrix.
    posterior = idata.posterior
    baseline_hat = _ppc_v2_posterior_mean(
        posterior, "feature_baseline", ("feature",)
    )
    bmi_hat = _ppc_v2_posterior_mean(
        posterior, "feature_bmi_contrast", ("bmi_contrast", "feature")
    )
    covariate_hat = _ppc_v2_posterior_mean(
        posterior, "feature_covariate_slope", ("covariate", "feature")
    )
    patient_hat = _ppc_v2_posterior_mean(
        posterior, "patient_global_effect", ("patient",)
    )
    mu_hat = (
        baseline_hat[feature_idx]
        + np.einsum(
            "ok,ko->o", group_indicators, bmi_hat[:, feature_idx], optimize=True
        )
        + patient_hat[patient_idx]
        + np.einsum(
            "oc,co->o", covariates, covariate_hat[:, feature_idx], optimize=True
        )
    )

    observed_feature_matrix = observed[observation_grid]
    observed_residual_matrix = (observed - mu_hat)[observation_grid]
    observed_residual_sd, observed_studentized_tail = (
        _ppc_v2_residual_statistics(observed_residual_matrix[None, :, :])
    )
    observed_residual_sd = observed_residual_sd[0]
    observed_studentized_tail = observed_studentized_tail[0]

    observed_raw_dependence = np.array([
        _ppc_v2_dependence_count(
            observed_feature_matrix[None, :, :], columns,
            PPC_CORRELATION_THRESHOLD,
        )[0]
        for columns in feature_columns_by_celltype.values()
    ], dtype=np.int32)
    observed_residual_dependence = np.array([
        _ppc_v2_dependence_count(
            observed_residual_matrix[None, :, :], columns,
            PPC_CORRELATION_THRESHOLD,
        )[0]
        for columns in feature_columns_by_celltype.values()
    ], dtype=np.int32)

    replicate_residual_sd = np.empty(
        (requested_joint, n_features), dtype=np.float32
    )
    replicate_studentized_tail = np.empty(
        (requested_joint, n_features), dtype=np.float32
    )
    replicate_raw_dependence = np.empty(
        (requested_joint, n_celltypes), dtype=np.int32
    )
    replicate_residual_dependence = np.empty(
        (requested_joint, n_celltypes), dtype=np.int32
    )
    replicate_chain = np.empty(requested_joint, dtype=np.int16)
    replicate_draw_position = np.empty(requested_joint, dtype=np.int32)
    predictive_le_count = np.zeros(n_observations, dtype=np.int32)

    # Allow room for y_rep, mu, and feature-by-patient working arrays.
    bytes_per_position = max(1, 4 * 8 * n_observations * n_chains)
    positions_per_batch = max(
        1, int(PPC_V2_BATCH_MB * 1024**2 // bytes_per_position)
    )
    n_batches = math.ceil(per_chain / positions_per_batch)
    print(
        f"Corrected PPC v2: {requested_joint} joint datasets "
        f"in {n_batches} memory-bounded batches"
    )

    cursor = 0
    batch_records = []
    for batch_number, start in enumerate(
        range(0, per_chain, positions_per_batch)
    ):
        if batch_number == 0 or (batch_number + 1) % 5 == 0 or (batch_number + 1) == n_batches:
            print(f"PPC v2 batch {batch_number + 1}/{n_batches}")
        positions = selected_positions[start:start + positions_per_batch]
        posterior_batch_dataset = posterior.isel(draw=positions)
        posterior_batch = az.InferenceData(posterior=posterior_batch_dataset)
        seed = RUN_SEED + 70_000 + batch_number
        with model:
            prediction = pm.sample_posterior_predictive(
                posterior_batch,
                var_names=["y_obs"],
                random_seed=seed,
                return_inferencedata=False,
                progressbar=False,
            )

        replicated = np.asarray(prediction["y_obs"], dtype=float)
        expected_shape = (n_chains, len(positions), n_observations)
        if replicated.shape != expected_shape:
            raise RuntimeError(
                f"Unexpected PPC shape {replicated.shape}; expected {expected_shape}"
            )
        replicated = replicated.reshape(-1, n_observations)
        n_batch = len(replicated)
        stop = cursor + n_batch

        baseline_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "feature_baseline", ("feature",)
        )
        bmi_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "feature_bmi_contrast",
            ("bmi_contrast", "feature"),
        )
        covariate_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset,
            "feature_covariate_slope",
            ("covariate", "feature"),
        )
        patient_draw = _ppc_v2_flat_draws(
            posterior_batch_dataset, "patient_global_effect", ("patient",)
        )
        mu_draw = (
            baseline_draw[:, feature_idx]
            + np.einsum(
                "ok,bko->bo", group_indicators,
                bmi_draw[:, :, feature_idx], optimize=True,
            )
            + patient_draw[:, patient_idx]
            + np.einsum(
                "oc,bco->bo",
                covariates,
                covariate_draw[:, :, feature_idx],
                optimize=True,
            )
        )

        predictive_le_count += np.sum(
            replicated <= observed[None, :], axis=0
        ).astype(np.int32)

        raw_feature_matrix = replicated[:, observation_grid]
        for j, columns in feature_columns_by_celltype.items():
            replicate_raw_dependence[cursor:stop, j] = (
                _ppc_v2_dependence_count(
                    raw_feature_matrix, columns, PPC_CORRELATION_THRESHOLD
                )
            )
        del raw_feature_matrix

        # Conditional residuals: y_rep - mu for the same posterior draw.
        replicated -= mu_draw
        residual_feature_matrix = replicated[:, observation_grid]
        batch_sd, batch_tail = _ppc_v2_residual_statistics(
            residual_feature_matrix
        )
        replicate_residual_sd[cursor:stop] = batch_sd
        replicate_studentized_tail[cursor:stop] = batch_tail
        for j, columns in feature_columns_by_celltype.items():
            replicate_residual_dependence[cursor:stop, j] = (
                _ppc_v2_dependence_count(
                    residual_feature_matrix,
                    columns,
                    PPC_CORRELATION_THRESHOLD,
                )
            )

        replicate_chain[cursor:stop] = np.repeat(
            np.arange(n_chains), len(positions)
        )
        replicate_draw_position[cursor:stop] = np.tile(
            positions, n_chains
        )
        batch_records.append({
            "batch": batch_number,
            "seed": seed,
            "positions_per_chain": len(positions),
            "joint_datasets": n_batch,
        })
        cursor = stop

        del (
            posterior_batch, posterior_batch_dataset, prediction, replicated,
            residual_feature_matrix, mu_draw, baseline_draw, bmi_draw,
            covariate_draw, patient_draw, batch_sd, batch_tail,
        )
        gc.collect()

    if cursor != requested_joint:
        raise RuntimeError(f"Generated {cursor}; expected {requested_joint}")

    calibration_mask = np.zeros(requested_joint, dtype=bool)
    evaluation_mask = np.zeros(requested_joint, dtype=bool)
    for chain in range(n_chains):
        indices = np.flatnonzero(replicate_chain == chain)
        calibration_mask[indices[::2]] = True
        evaluation_mask[indices[1::2]] = True
    if (
        calibration_mask.sum() != requested_joint // 2
        or evaluation_mask.sum() != requested_joint // 2
        or np.any(calibration_mask & evaluation_mask)
    ):
        raise RuntimeError("The balanced PPC calibration/evaluation split failed")

    inventory = feature_inventory.copy().set_index("global_feature").loc[
        model_arrays["features"]
    ].reset_index()
    feature_checks = inventory.copy()
    feature_checks["observed_residual_sd"] = observed_residual_sd
    feature_checks["replicated_residual_sd_median"] = np.median(
        replicate_residual_sd, axis=0
    )
    feature_checks["observed_studentized_max_abs"] = observed_studentized_tail
    feature_checks["replicated_studentized_max_abs_median"] = np.median(
        replicate_studentized_tail, axis=0
    )

    domain_rows = []

    def add_count_rows(domain, observed_celltype, replicated_celltype):
        labels = ["__GLOBAL__", *model_arrays["celltypes"]]
        observed_all = np.r_[observed_celltype.sum(), observed_celltype]
        replicated_all = np.column_stack([
            replicated_celltype.sum(axis=1), replicated_celltype
        ])
        for j, label in enumerate(labels):
            reference = replicated_all[:, j]
            probability = _ppc_v2_upper_p(reference, observed_all[j])
            mcse = float(
                np.sqrt(probability * (1 - probability) / len(reference))
            )
            domain_rows.append({
                "domain": domain,
                "global_celltype": label,
                "observed_count": int(observed_all[j]),
                "replicated_q025": float(np.quantile(reference, 0.025)),
                "replicated_median": float(np.median(reference)),
                "replicated_q975": float(np.quantile(reference, 0.975)),
                "upper_tail_p": probability,
                "mcse_approx": mcse,
                "review_flag_p_le_0_025": bool(probability <= PPC_V2_ALPHA),
            })

    def calibrated_feature_domain(domain, observed_statistic, replicated_statistic):
        calibration = replicated_statistic[calibration_mask]
        evaluation = replicated_statistic[evaluation_mask]
        low, high = np.quantile(calibration, [0.025, 0.975], axis=0)
        observed_flags = (
            (observed_statistic < low) | (observed_statistic > high)
        )
        evaluation_flags = (
            (evaluation < low[None, :]) | (evaluation > high[None, :])
        )
        observed_celltype = np.array([
            observed_flags[feature_columns_by_celltype[j]].sum()
            for j in range(n_celltypes)
        ], dtype=int)
        evaluation_celltype = np.column_stack([
            evaluation_flags[:, feature_columns_by_celltype[j]].sum(axis=1)
            for j in range(n_celltypes)
        ])
        add_count_rows(domain, observed_celltype, evaluation_celltype)
        return low, high, observed_flags

    dispersion_low, dispersion_high, dispersion_flags = (
        calibrated_feature_domain(
            "residual_dispersion",
            observed_residual_sd,
            replicate_residual_sd,
        )
    )
    tail_low, tail_high, tail_flags = calibrated_feature_domain(
        "studentized_residual_tails",
        observed_studentized_tail,
        replicate_studentized_tail,
    )
    feature_checks["residual_sd_calibration_q025"] = dispersion_low
    feature_checks["residual_sd_calibration_q975"] = dispersion_high
    feature_checks["residual_dispersion_outside"] = dispersion_flags
    feature_checks["studentized_tail_calibration_q025"] = tail_low
    feature_checks["studentized_tail_calibration_q975"] = tail_high
    feature_checks["studentized_tail_outside"] = tail_flags

    add_count_rows(
        "raw_outcome_dependence",
        observed_raw_dependence,
        replicate_raw_dependence[evaluation_mask],
    )
    add_count_rows(
        "conditional_residual_dependence",
        observed_residual_dependence,
        replicate_residual_dependence[evaluation_mask],
    )

    predictive_cdf = (1 + predictive_le_count) / (requested_joint + 1)
    covered_90 = (predictive_cdf >= 0.05) & (predictive_cdf <= 0.95)
    feature_checks["predictive_90pct_coverage"] = np.array([
        covered_90[observation_grid[j]].mean() for j in range(n_features)
    ])

    domain_checks = pd.DataFrame(domain_rows)
    global_checks = domain_checks[
        domain_checks["global_celltype"].eq("__GLOBAL__")
    ].copy()
    review_domains = global_checks.loc[
        global_checks["review_flag_p_le_0_025"], "domain"
    ].tolist()
    global_tail_row = global_checks.loc[
        global_checks["domain"].eq("studentized_residual_tails")
    ]
    if len(global_tail_row) != 1:
        raise RuntimeError("Expected exactly one global tail-check row")
    tail_adequacy_pass = bool(
        not global_tail_row.iloc[0]["review_flag_p_le_0_025"]
    )

    feature_path = OUTPUT_DIR / "ppc_v2_feature_checks.csv"
    domain_path = OUTPUT_DIR / "ppc_v2_domain_checks.csv"
    discrepancy_path = OUTPUT_DIR / "ppc_v2_joint_discrepancies.npz"
    summary_path = OUTPUT_DIR / "ppc_v2_summary.json"
    atomic_csv(feature_checks, feature_path)
    atomic_csv(domain_checks, domain_path)
    _ppc_v2_atomic_npz(
        discrepancy_path,
        residual_sd=replicate_residual_sd,
        studentized_max_abs=replicate_studentized_tail,
        raw_dependence_count=replicate_raw_dependence,
        residual_dependence_count=replicate_residual_dependence,
        chain=replicate_chain,
        draw_position=replicate_draw_position,
        calibration_mask=calibration_mask,
        evaluation_mask=evaluation_mask,
        observed_residual_sd=observed_residual_sd,
        observed_studentized_max_abs=observed_studentized_tail,
        observed_raw_dependence=observed_raw_dependence,
        observed_residual_dependence=observed_residual_dependence,
    )

    summary = {
        "ppc_version": PPC_V2_VERSION,
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "nu_scope": "celltype_within_separate_compartment",
        "joint_draws": requested_joint,
        "calibration_draws": int(calibration_mask.sum()),
        "evaluation_draws": int(evaluation_mask.sum()),
        "n_features": n_features,
        "n_celltypes": n_celltypes,
        "correlation_threshold": PPC_CORRELATION_THRESHOLD,
        "observed_raw_high_correlation_pairs": int(
            observed_raw_dependence.sum()
        ),
        "observed_residual_high_correlation_pairs": int(
            observed_residual_dependence.sum()
        ),
        "median_predictive_90pct_coverage": float(
            feature_checks["predictive_90pct_coverage"].median()
        ),
        "minimum_predictive_90pct_coverage": float(
            feature_checks["predictive_90pct_coverage"].min()
        ),
        "global_domains_flagged_for_review": review_domains,
        "tail_adequacy_pass": tail_adequacy_pass,
        "decision": (
            "CELLTYPE_NU_TAIL_CHECK_PASSED"
            if tail_adequacy_pass
            else "CELLTYPE_NU_TAIL_MISMATCH_REMAINS"
        ),
        "important": (
            "This categorical analysis is a prespecified secondary response-shape analysis. "
            "Its PPC is evaluated independently and does "
            "not authorize BMI-effect reveal by itself."
        ),
        "feature_checks_sha256": sha256_file(feature_path),
        "domain_checks_sha256": sha256_file(domain_path),
        "joint_discrepancies_sha256": sha256_file(discrepancy_path),
        "batches": batch_records,
    }
    atomic_json(summary, summary_path)

    display(domain_checks)
    print(json.dumps(summary, indent=2))
    print("Saved corrected PPC v2 outputs in:", OUTPUT_DIR)
else:
    print("Corrected PPC v2 disabled; set WORKFLOW_STAGE='ppc'.")


In [ ]:
# 7a. Review categorical PPC and record the adequacy decision
if RUN_PPC_REVIEW:
    summary_path = OUTPUT_DIR / "ppc_v2_summary.json"
    domain_path = OUTPUT_DIR / "ppc_v2_domain_checks.csv"
    feature_path = OUTPUT_DIR / "ppc_v2_feature_checks.csv"
    diagnostics_path = OUTPUT_DIR / "run_diagnostics.json"
    for path in (summary_path, domain_path, feature_path, diagnostics_path):
        if not path.exists():
            raise FileNotFoundError(f"Missing {path.name}; run the required earlier stage")
    saved_ppc = json.loads(summary_path.read_text(encoding="utf-8"))
    diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    identity_checks = {
        "trace": saved_ppc.get("posterior_trace_sha256") == sha256_file(TRACE_PATH),
        "model": saved_ppc.get("model_fingerprint") == MODEL_FINGERPRINT,
        "lock": saved_ppc.get("analysis_lock_sha256") == ANALYSIS_LOCK_SHA256,
        "diagnostics": saved_ppc.get("diagnostics_sha256") == sha256_file(diagnostics_path),
        "release": NOTEBOOK_RELEASE.get("status") == "verified",
        "convergence": diagnostics.get("current_profile_gate_pass") is True,
    }
    failed = [name for name, passed in identity_checks.items() if not passed]
    if failed:
        raise RuntimeError(f"PPC review identity failed: {failed}")

    automated_tail_pass = saved_ppc.get("tail_adequacy_pass") is True
    justification = PPC_OVERRIDE_JUSTIFICATION.strip()
    override_valid = bool(
        PPC_OVERRIDE_APPROVED and len(justification) >= 30
    )
    ppc_adequacy_pass = bool(automated_tail_pass or override_valid)
    if automated_tail_pass:
        decision = "PASS_AUTOMATED_TAIL_CHECK"
    elif override_valid:
        decision = "PASS_WITH_DOCUMENTED_PPC_LIMITATION"
    else:
        decision = "WITHHELD_TAIL_REVIEW_REQUIRED"
    adequacy = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "notebook_release_status": NOTEBOOK_RELEASE["status"],
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "ppc_summary_sha256": sha256_file(summary_path),
        "ppc_feature_checks_sha256": sha256_file(feature_path),
        "ppc_domain_checks_sha256": sha256_file(domain_path),
        "automated_tail_adequacy_pass": automated_tail_pass,
        "review_domains": saved_ppc.get("global_domains_flagged_for_review", []),
        "manual_override_approved": bool(PPC_OVERRIDE_APPROVED),
        "manual_override_justification": justification if override_valid else None,
        "adequacy_decision": decision,
        "ppc_adequacy_pass": ppc_adequacy_pass,
        "important": (
            "A manual override records a limitation; it does not transform a failed "
            "posterior-predictive check into evidence of perfect fit."
        ),
    }
    atomic_json(adequacy, OUTPUT_DIR / "ppc_adequacy.json")
    display(Markdown(
        f"### Categorical PPC decision: `{decision}`\n\n"
        f"- automated tail pass: `{automated_tail_pass}`\n"
        f"- final adequacy gate: `{ppc_adequacy_pass}`\n"
        f"- review domains: `{adequacy['review_domains']}`"
    ))
    display(pd.read_csv(domain_path).sort_values(
        ["domain", "upper_tail_p", "global_celltype"]
    ))
    display(az.summary(idata, var_names=["nu"], kind="all", round_to=4))
    if not ppc_adequacy_pass:
        raise RuntimeError(
            "PPC reveal remains blocked. Review the tail mismatch. If scientifically "
            "defensible for this secondary analysis, set PPC_OVERRIDE_APPROVED=True "
            "and provide at least 30 characters of justification."
        )
else:
    print("PPC review disabled; set WORKFLOW_STAGE='ppc_review'.")


In [ ]:
# No-refit patient influence audit for categorical PPC tail events

if idata is None:
    raise RuntimeError("The saved posterior trace must be loaded.")

# 1. Reconstruct posterior-mean fitted values and residuals

data = model_arrays["data"].copy()
posterior = idata.posterior

baseline_hat = np.asarray(
    posterior["feature_baseline"]
    .mean(dim=("chain", "draw"))
    .transpose("feature"),
    dtype=float,
)

bmi_hat = np.asarray(
    posterior["feature_bmi_contrast"]
    .mean(dim=("chain", "draw"))
    .transpose("bmi_contrast", "feature"),
    dtype=float,
)

covariate_hat = np.asarray(
    posterior["feature_covariate_slope"]
    .mean(dim=("chain", "draw"))
    .transpose("covariate", "feature"),
    dtype=float,
)

patient_hat = np.asarray(
    posterior["patient_global_effect"]
    .mean(dim=("chain", "draw"))
    .transpose("patient"),
    dtype=float,
)

feature_idx = data["feature_idx"].to_numpy(int)
patient_idx = data["patient_idx"].to_numpy(int)

group_matrix = data[
    ["is_overweight", "is_obese"]
].to_numpy(float)

covariate_matrix = data[
    ["age_z", "sex_centered"]
].to_numpy(float)

mu_hat = (
    baseline_hat[feature_idx]
    + np.einsum(
        "ok,ko->o",
        group_matrix,
        bmi_hat[:, feature_idx],
        optimize=True,
    )
    + np.einsum(
        "oc,co->o",
        covariate_matrix,
        covariate_hat[:, feature_idx],
        optimize=True,
    )
    + patient_hat[patient_idx]
)

data["posterior_mean"] = mu_hat
data["residual"] = data["score_model"].to_numpy(float) - mu_hat

# Studentize residuals within each pathway across patients.
data["residual_centered"] = data["residual"] - data.groupby(
    "global_feature"
)["residual"].transform("mean")

data["residual_sd"] = data.groupby(
    "global_feature"
)["residual_centered"].transform("std")

if (
    data["residual_sd"].isna().any()
    or (data["residual_sd"] <= 0).any()
):
    raise RuntimeError("Invalid pathway residual SD.")

data["abs_studentized_residual"] = (
    data["residual_centered"].abs() / data["residual_sd"]
)

# 2. Obtain the PPC pathway-specific tail thresholds

feature_checks = pd.read_csv(
    OUTPUT_DIR / "ppc_v2_feature_checks.csv"
)

required_columns = {
    "global_feature",
    "studentized_tail_calibration_q975",
    "studentized_tail_outside",
}

missing = required_columns - set(feature_checks.columns)
if missing:
    raise RuntimeError(
        f"PPC feature-check columns missing: {sorted(missing)}"
    )

tail_threshold = feature_checks.set_index(
    "global_feature"
)["studentized_tail_calibration_q975"]

tail_flag = feature_checks.set_index(
    "global_feature"
)["studentized_tail_outside"]

data["tail_threshold"] = data["global_feature"].map(tail_threshold)
data["pathway_tail_flag"] = data["global_feature"].map(tail_flag).astype(bool)

# For every flagged pathway, identify the patient producing its
# largest absolute studentized residual.
flagged = data[data["pathway_tail_flag"]].copy()

largest_rows = flagged.loc[
    flagged.groupby("global_feature")[
        "abs_studentized_residual"
    ].idxmax()
].copy()

largest_rows["exceeds_ppc_threshold"] = (
    largest_rows["abs_studentized_residual"]
    > largest_rows["tail_threshold"]
)

# 3. Compute patient-design leverage

patient_table = (
    data[
        [
            "patient_idx",
            "base_sample_id",
            "bmi",
            "bmi_z",
            "bmi_group",
            "age_z",
            "sex_centered",
        ]
    ]
    .drop_duplicates("patient_idx")
    .sort_values("patient_idx")
    .reset_index(drop=True)
)

X = np.column_stack(
    [
        np.ones(len(patient_table)),
        patient_table["is_overweight"].to_numpy(float)
        if "is_overweight" in patient_table.columns
        else patient_table["bmi_group"].eq("overweight").to_numpy(float),
        patient_table["is_obese"].to_numpy(float)
        if "is_obese" in patient_table.columns
        else patient_table["bmi_group"].eq("obese").to_numpy(float),
        patient_table["age_z"].to_numpy(float),
        patient_table["sex_centered"].to_numpy(float),
    ]
)

hat = X @ np.linalg.pinv(X.T @ X) @ X.T
patient_table["design_leverage"] = np.diag(hat)

p = X.shape[1]
n = X.shape[0]
high_leverage_threshold = 2 * p / n

patient_table["high_leverage"] = (
    patient_table["design_leverage"] > high_leverage_threshold
)

patient_table["extreme_bmi"] = (
    patient_table["bmi_z"].abs() >= 2
)

patient_table["leverage_percentile"] = (
    patient_table["design_leverage"].rank(pct=True) * 100
)

# 4. Join patient characteristics to the tail events

audit_columns = [
    "patient_idx",
    "base_sample_id",
    "bmi",
    "bmi_z",
    "bmi_group",
    "age_z",
    "sex_centered",
    "design_leverage",
    "leverage_percentile",
    "high_leverage",
    "extreme_bmi",
]

# Remove existing patient-level columns before merging, preventing
# pandas from creating bmi_x/bmi_y and related duplicate columns.
patient_variables = [
    "bmi",
    "bmi_z",
    "bmi_group",
    "age_z",
    "sex_centered",
    "design_leverage",
    "leverage_percentile",
    "high_leverage",
    "extreme_bmi",
]

audit = (
    largest_rows.drop(
        columns=patient_variables,
        errors="ignore",
    )
    .merge(
        patient_table[audit_columns],
        on=["patient_idx", "base_sample_id"],
        how="left",
        validate="many_to_one",
    )
)

required_audit_columns = [
    "base_sample_id",
    "CellType",
    "Signature",
    "global_feature",
    "bmi_group",
    "bmi",
    "bmi_z",
    "age_z",
    "sex_centered",
    "residual_centered",
    "abs_studentized_residual",
    "tail_threshold",
    "exceeds_ppc_threshold",
    "design_leverage",
    "leverage_percentile",
    "high_leverage",
    "extreme_bmi",
]

missing_audit_columns = [
    column
    for column in required_audit_columns
    if column not in audit.columns
]

if missing_audit_columns:
    raise RuntimeError(
        f"Influence-audit columns missing: {missing_audit_columns}"
    )

audit = audit[required_audit_columns].sort_values(
    ["CellType", "abs_studentized_residual"],
    ascending=[True, False],
)

# Principal review: the two cell types with flagged tail counts.
focus = audit[
    audit["CellType"].isin(
        ["CD4+ T CELLS", "DENDRITIC_CELLS"]
    )
].copy()

if focus.empty:
    raise RuntimeError(
        "No CD4 or dendritic tail events were found. "
        f"Available cell-type labels: {sorted(audit['CellType'].unique())}"
    )

# Summarize whether multiple flagged pathways point to the same patient.
patient_summary = (
    focus.groupby(
        [
            "base_sample_id",
            "bmi_group",
            "bmi",
            "bmi_z",
            "design_leverage",
            "leverage_percentile",
            "high_leverage",
            "extreme_bmi",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        tail_pathways=("global_feature", "nunique"),
        affected_celltypes=("CellType", "nunique"),
        maximum_studentized_residual=(
            "abs_studentized_residual",
            "max",
        ),
    )
    .sort_values(
        ["tail_pathways", "maximum_studentized_residual"],
        ascending=[False, False],
    )
)

group_summary = (
    focus.groupby("bmi_group", observed=True)
    .agg(
        tail_pathways=("global_feature", "nunique"),
        unique_patients=("base_sample_id", "nunique"),
        high_leverage_events=("high_leverage", "sum"),
        extreme_bmi_events=("extreme_bmi", "sum"),
    )
    .reindex(["normal_weight", "overweight", "obese"])
    .fillna(0)
)

print(f"High-leverage threshold = {high_leverage_threshold:.4f}")
print(f"Flagged pathways across all cell types = {len(audit)}")
print(f"Flagged CD4/dendritic pathways = {len(focus)}")
print(
    "Unique patients producing CD4/dendritic maxima = "
    f"{focus['base_sample_id'].nunique()}"
)

display(Markdown("### CD4 and dendritic tail-event details"))
display(focus)

display(Markdown("### Patient concentration summary"))
display(patient_summary)

display(Markdown("### BMI-group distribution"))
display(group_summary)

# Save the audit separately. No model or trace file is changed.
audit_directory = OUTPUT_DIR / "influence_audit"
audit_directory.mkdir(parents=True, exist_ok=True)

audit.to_csv(
    audit_directory / "all_flagged_pathway_patient_maxima.csv",
    index=False,
)

focus.to_csv(
    audit_directory / "cd4_dendritic_tail_event_audit.csv",
    index=False,
)

patient_summary.to_csv(
    audit_directory / "cd4_dendritic_patient_summary.csv",
    index=False,
)

group_summary.to_csv(
    audit_directory / "cd4_dendritic_bmi_group_summary.csv"
)

print("Saved no-refit influence audit in:", audit_directory)


In [ ]:
# 9. Reveal categorical contrasts with within-contrast and global ROPE-BFDR

def _az_da_stat(da, statistic: str, method: str):
    dataset = xr.Dataset({"metric": da.astype(float)})
    function = az.ess if statistic == "ess" else az.mcse
    return function(dataset, var_names=["metric"], method=method)["metric"]

def effective_sample_boundary_guard(mean, n_eff, alpha=MC_ALPHA):
    """Conservative Monte Carlo boundary guard, not an exact data CI."""
    n_eff = max(1.0, float(n_eff))
    mean = float(np.clip(mean, 0, 1))
    x = mean * n_eff
    lower = 0.0 if x <= 0 else float(beta_dist.ppf(alpha, x, n_eff - x + 1))
    upper = 1.0 if x >= n_eff else float(
        beta_dist.ppf(1 - alpha, x + 1, n_eff - x)
    )
    return lower, upper

def rope_indicator_metrics(beta, effect_dim: str, id_column: str, rope: float):
    beta = beta.transpose("chain", "draw", effect_dim)
    indicator = (np.abs(beta) <= float(rope)).astype(np.float32).rename(
        "rope_indicator"
    )
    probability = indicator.mean(("chain", "draw")).to_numpy()
    ess_raw = _az_da_stat(indicator, "ess", "bulk").to_numpy()
    mcse_raw = _az_da_stat(indicator, "mcse", "mean").to_numpy()
    beta_bulk = _az_da_stat(beta, "ess", "bulk").to_numpy()
    beta_tail = _az_da_stat(beta, "ess", "tail").to_numpy()
    n_total = beta.sizes["chain"] * beta.sizes["draw"]
    fallback = np.minimum(beta_bulk, beta_tail)
    ess = np.where(np.isfinite(ess_raw) & (ess_raw > 0), ess_raw, fallback)
    ess = np.minimum(ess, n_total)
    z_value = norm.ppf(1 - MC_ALPHA)
    rows = []
    for j, name in enumerate(beta.coords[effect_dim].to_numpy()):
        lower, upper = effective_sample_boundary_guard(probability[j], ess[j])
        raw = float(mcse_raw[j]) if np.isfinite(mcse_raw[j]) else 0.0
        guarded = max(
            raw,
            (upper - probability[j]) / z_value,
            (probability[j] - lower) / z_value,
        )
        rows.append({
            id_column: str(name),
            "local_null_probability": float(probability[j]),
            "indicator_ess": float(ess[j]),
            "indicator_mcse_raw": raw,
            "indicator_mcse_guarded": float(guarded),
            "local_null_mc_lower": lower,
            "local_null_mc_upper": upper,
        })
    return indicator, pd.DataFrame(rows)

def deterministic_bfdr(local_null, identifiers, q=Q_LEVEL):
    values = np.asarray(local_null, float)
    identifiers = np.asarray(identifiers, str)
    if not len(values) or not np.isfinite(values).all():
        raise ValueError("Invalid local-null vector")
    order = np.lexsort((identifiers, values))
    cumulative = np.cumsum(values[order]) / np.arange(1, len(values) + 1)
    accepted = np.flatnonzero(cumulative <= q)
    k = int(accepted[-1] + 1) if len(accepted) else 0
    selected = np.zeros(len(values), dtype=bool)
    selected[order[:k]] = True
    return selected, order, cumulative, k

def bounded_series_mc(series, fallback_ess, alpha=MC_ALPHA):
    mean = float(series.mean())
    n_total = series.sizes["chain"] * series.sizes["draw"]
    ess = float(np.asarray(_az_da_stat(series.rename("metric"), "ess", "bulk")))
    mcse = float(np.asarray(_az_da_stat(series.rename("metric"), "mcse", "mean")))
    if not np.isfinite(ess) or ess <= 0:
        ess = float(fallback_ess)
    ess = min(ess, n_total)
    if not np.isfinite(mcse):
        mcse = 0.0
    guard_low, guard_high = effective_sample_boundary_guard(mean, ess, alpha)
    z_value = norm.ppf(1 - alpha)
    lower = max(0.0, min(mean - z_value * mcse, guard_low))
    upper = min(1.0, max(mean + z_value * mcse, guard_high))
    guarded_mcse = max(
        mcse, (upper - mean) / z_value, (mean - lower) / z_value
    )
    return {
        "mean": mean,
        "ess": ess,
        "mcse_raw": mcse,
        "mcse_guarded": guarded_mcse,
        "mc_lower": lower,
        "mc_upper": upper,
    }

def selection_sets(mask, identifiers, cluster_ids):
    return (
        frozenset(np.asarray(identifiers, str)[mask]),
        frozenset(np.asarray(cluster_ids, str)[mask]),
    )

def assess_rope_bfdr(
    beta, effect_dim, id_column, identifiers, cluster_ids, rope, q=Q_LEVEL
):
    if beta.sizes["chain"] < 4:
        raise ValueError("Reveal requires at least four chains")
    indicator, metrics = rope_indicator_metrics(beta, effect_dim, id_column, rope)
    trace_ids = np.asarray(indicator.coords[effect_dim].to_numpy(), str)
    identifiers = np.asarray(identifiers, str)
    raw_cluster_ids = np.asarray(cluster_ids, dtype=object)
    if pd.isna(raw_cluster_ids).any():
        raise ValueError("Missing correlation-cluster ID")
    cluster_ids = raw_cluster_ids.astype(str)
    if not np.array_equal(trace_ids, identifiers):
        raise ValueError("Effect order differs from posterior coordinates")
    if (
        np.any(np.char.strip(cluster_ids) == "")
        or np.any(np.char.lower(cluster_ids) == "nan")
    ):
        raise ValueError("Empty/invalid correlation-cluster ID")

    probability = metrics["local_null_probability"].to_numpy()
    mask, order, cumulative, k = deterministic_bfdr(probability, identifiers, q)
    next_index = None if k == len(probability) else int(order[k])
    critical = mask | (probability <= 0.10)
    if next_index is not None:
        critical[next_index] = True
    critical_metrics = metrics.loc[critical]
    indicator_precision_pass = bool(
        (critical_metrics["indicator_ess"] >= ROPE_INDICATOR_ESS_MIN).all()
        and (
            critical_metrics["indicator_mcse_guarded"]
            <= ROPE_LOCAL_NULL_MCSE_MAX
        ).all()
    )
    if k:
        selected_series = indicator.isel(
            {effect_dim: np.flatnonzero(mask)}
        ).mean(effect_dim)
        selected_mc = bounded_series_mc(
            selected_series, critical_metrics["indicator_ess"].min()
        )
        selected_bound_pass = bool(
            selected_mc["mc_upper"] <= q
            and selected_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        selected_mc = {
            "mean": 0.0, "ess": None, "mcse_raw": 0.0,
            "mcse_guarded": 0.0, "mc_lower": 0.0, "mc_upper": 0.0,
        }
        selected_bound_pass = True
    if next_index is not None:
        prefix = order[:k + 1]
        next_series = indicator.isel({effect_dim: prefix}).mean(effect_dim)
        next_mc = bounded_series_mc(
            next_series, metrics.iloc[prefix]["indicator_ess"].min()
        )
        next_bound_pass = bool(
            next_mc["mc_lower"] > q
            and next_mc["mcse_guarded"] <= BFDR_MCSE_MAX
        )
    else:
        next_mc = None
        next_bound_pass = True

    full_identifiers, full_clusters = selection_sets(mask, identifiers, cluster_ids)
    loo_rows = []
    for omitted in range(indicator.sizes["chain"]):
        keep = [j for j in range(indicator.sizes["chain"]) if j != omitted]
        loo_probability = indicator.isel(chain=keep).mean(
            ("chain", "draw")
        ).to_numpy()
        loo_mask, _, _, _ = deterministic_bfdr(loo_probability, identifiers, q)
        identifier_set, cluster_set = selection_sets(
            loo_mask, identifiers, cluster_ids
        )
        loo_rows.append({
            "omitted_chain_position": omitted,
            "identifier_set_identical": identifier_set == full_identifiers,
            "cluster_set_identical": cluster_set == full_clusters,
        })
    loo = pd.DataFrame(loo_rows)
    loo_pass = bool(loo[
        ["identifier_set_identical", "cluster_set_identical"]
    ].to_numpy().all())
    ranks = np.empty(len(probability), int)
    ranks[order] = np.arange(1, len(probability) + 1)
    cumulative_by_effect = np.full(len(probability), np.nan)
    cumulative_by_effect[order] = cumulative
    metrics["bfdr_rank"] = ranks
    metrics["cumulative_bfdr_at_rank"] = cumulative_by_effect
    metrics["bfdr_selected"] = mask
    return {
        "pass": bool(
            indicator_precision_pass and selected_bound_pass
            and next_bound_pass and loo_pass
        ),
        "rope": float(rope),
        "n_selected": k,
        "indicator_precision_pass": indicator_precision_pass,
        "selected_mc": selected_mc,
        "selected_bound_pass": selected_bound_pass,
        "next_prefix_mc": next_mc,
        "next_bound_pass": next_bound_pass,
        "loo_pass": loo_pass,
        "effect_metrics": metrics,
        "loo": loo,
        "minimum_critical_indicator_ess": float(
            critical_metrics["indicator_ess"].min()
        ),
        "maximum_critical_indicator_mcse": float(
            critical_metrics["indicator_mcse_guarded"].max()
        ),
    }

def posterior_summary_frame(beta, effect_dim: str, id_column: str) -> pd.DataFrame:
    beta = beta.transpose("chain", "draw", effect_dim)
    samples = beta.stack(sample=("chain", "draw")).transpose(
        effect_dim, "sample"
    ).to_numpy()
    dataset = xr.Dataset({"effect": beta})
    hdi = az.hdi(dataset, var_names=["effect"], hdi_prob=0.95)["effect"].transpose(
        effect_dim, "hdi"
    ).to_numpy()
    rhat = az.rhat(dataset, var_names=["effect"])["effect"].to_numpy()
    ess_bulk = az.ess(
        dataset, var_names=["effect"], method="bulk"
    )["effect"].to_numpy()
    ess_tail = az.ess(
        dataset, var_names=["effect"], method="tail"
    )["effect"].to_numpy()
    return pd.DataFrame({
        id_column: beta.coords[effect_dim].astype(str).to_numpy(),
        "posterior_mean": samples.mean(axis=1),
        "posterior_median": np.median(samples, axis=1),
        "posterior_sd": samples.std(axis=1, ddof=1),
        "hdi_low": hdi[:, 0],
        "hdi_high": hdi[:, 1],
        "posterior_prob_positive": np.mean(samples > 0, axis=1),
        "posterior_prob_negative": np.mean(samples < 0, axis=1),
        "r_hat": rhat,
        "ess_bulk": ess_bulk,
        "ess_tail": ess_tail,
    })

def hedges_g(first, second):
    first = np.asarray(first, float)
    second = np.asarray(second, float)
    n1, n2 = len(first), len(second)
    pooled_variance = (
        (n1 - 1) * first.var(ddof=1) + (n2 - 1) * second.var(ddof=1)
    ) / (n1 + n2 - 2)
    if not np.isfinite(pooled_variance) or pooled_variance <= 0:
        return np.nan
    d = (first.mean() - second.mean()) / np.sqrt(pooled_variance)
    correction = 1 - 3 / (4 * (n1 + n2) - 9)
    return float(correction * d)

if RUN_RESULT_REVEAL:
    if not REVEAL_RESULTS_APPROVED:
        raise RuntimeError(
            "Set REVEAL_RESULTS_APPROVED=True only after convergence and PPC review."
        )
    if idata is None:
        raise ValueError("Reveal requires the loaded publication trace")
    diagnostics_path = OUTPUT_DIR / "run_diagnostics.json"
    ppc_path = OUTPUT_DIR / "ppc_adequacy.json"
    if not diagnostics_path.exists() or not ppc_path.exists():
        raise FileNotFoundError("Publication diagnostics and reviewed PPC are required")
    diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
    ppc = json.loads(ppc_path.read_text(encoding="utf-8"))
    reveal_checks = {
        "convergence": diagnostics.get("current_profile_gate_pass") is True,
        "ppc_adequacy": ppc.get("ppc_adequacy_pass") is True,
        "diagnostic_trace": diagnostics.get("posterior_trace_sha256") == sha256_file(TRACE_PATH),
        "ppc_trace": ppc.get("posterior_trace_sha256") == sha256_file(TRACE_PATH),
        "diagnostic_lock": diagnostics.get("analysis_lock_sha256") == ANALYSIS_LOCK_SHA256,
        "ppc_lock": ppc.get("analysis_lock_sha256") == ANALYSIS_LOCK_SHA256,
        "diagnostic_fingerprint": diagnostics.get("model_fingerprint") == MODEL_FINGERPRINT,
        "ppc_fingerprint": ppc.get("model_fingerprint") == MODEL_FINGERPRINT,
        "diagnostic_profile": diagnostics.get("profile") == PROFILE,
        "ppc_profile": ppc.get("profile") == PROFILE,
        "ppc_diagnostics": ppc.get("diagnostics_sha256") == sha256_file(diagnostics_path),
        "notebook_verified": NOTEBOOK_RELEASE.get("status") == "verified",
        "diagnostic_release": diagnostics.get("notebook_release_sha256") == NOTEBOOK_RELEASE["sha256"],
        "ppc_release": ppc.get("notebook_release_sha256") == NOTEBOOK_RELEASE["sha256"],
        "feature_inventory": preflight_manifest.get("feature_inventory_sha256") == sha256_file(
            OUTPUT_DIR / "feature_inventory.csv"
        ),
    }
    failed = [name for name, passed in reveal_checks.items() if not passed]
    if failed:
        raise RuntimeError(f"Result reveal blocked; failed checks: {failed}")

    inventory = feature_inventory.set_index("global_feature").loc[
        model_arrays["features"]
    ].reset_index()
    feature_ids = np.asarray(model_arrays["features"], str)
    cluster_ids = inventory["global_cluster_id"].astype(str).to_numpy()
    effect_arrays = {
        contrast: idata.posterior[variable].transpose("chain", "draw", "feature")
        for contrast, variable in FEATURE_EFFECT_VARIABLES.items()
    }

    within_assessments = {
        (contrast, rope): assess_rope_bfdr(
            effect_arrays[contrast], "feature", "global_feature",
            feature_ids, cluster_ids, rope,
        )
        for contrast in REPORTED_CONTRASTS
        for rope in ROPE_GRID
    }
    global_blocks = []
    global_ids = []
    global_clusters = []
    for contrast in REPORTED_CONTRASTS:
        ids = np.asarray([f"{contrast}||{name}" for name in feature_ids], str)
        block = effect_arrays[contrast].assign_coords(feature=ids)
        global_blocks.append(block)
        global_ids.extend(ids.tolist())
        global_clusters.extend([
            f"{contrast}||{cluster}" for cluster in cluster_ids
        ])
    global_beta = xr.concat(global_blocks, dim="feature")
    global_ids = np.asarray(global_ids, str)
    global_clusters = np.asarray(global_clusters, str)
    global_assessments = {
        rope: assess_rope_bfdr(
            global_beta, "feature", "hypothesis_id",
            global_ids, global_clusters, rope,
        )
        for rope in ROPE_GRID
    }
    all_precision_pass = bool(
        all(value["pass"] for value in within_assessments.values())
        and all(value["pass"] for value in global_assessments.values())
    )
    precision_gate = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "ppc_adequacy_sha256": sha256_file(ppc_path),
        "all_displayed_ropes_pass": all_precision_pass,
        "within_contrast": {
            f"{contrast}|{rope}": {
                "pass": value["pass"],
                "n_selected": value["n_selected"],
                "indicator_precision_pass": value["indicator_precision_pass"],
                "selected_bound_pass": value["selected_bound_pass"],
                "next_bound_pass": value["next_bound_pass"],
                "loo_pass": value["loo_pass"],
                "minimum_critical_indicator_ess": value["minimum_critical_indicator_ess"],
                "maximum_critical_indicator_mcse": value["maximum_critical_indicator_mcse"],
            }
            for (contrast, rope), value in within_assessments.items()
        },
        "global_three_contrast": {
            str(rope): {
                "pass": value["pass"],
                "n_selected": value["n_selected"],
                "indicator_precision_pass": value["indicator_precision_pass"],
                "selected_bound_pass": value["selected_bound_pass"],
                "next_bound_pass": value["next_bound_pass"],
                "loo_pass": value["loo_pass"],
                "minimum_critical_indicator_ess": value["minimum_critical_indicator_ess"],
                "maximum_critical_indicator_mcse": value["maximum_critical_indicator_mcse"],
            }
            for rope, value in global_assessments.items()
        },
        "selection_status": (
            "precision_pass" if all_precision_pass else "withheld_precision_failed"
        ),
    }
    atomic_json(precision_gate, OUTPUT_DIR / "reveal_precision_gate.json")
    if not all_precision_pass:
        if PROFILE == "publication":
            raise RuntimeError(
                "ROPE/BFDR Monte Carlo precision failed. publication_long is the "
                "only prespecified additional fit; no effect table was written."
            )
        raise RuntimeError(
            "publication_long still lacks ROPE/BFDR precision. Stop; no further "
            "automatic refit is authorized."
        )

    global_metric_lookup = {
        rope: assessment["effect_metrics"].set_index("hypothesis_id")
        for rope, assessment in global_assessments.items()
    }
    result_parts = []
    for contrast in REPORTED_CONTRASTS:
        beta = effect_arrays[contrast]
        part = inventory.merge(
            posterior_summary_frame(beta, "feature", "global_feature"),
            on="global_feature", how="left", validate="one_to_one",
        )
        part.insert(0, "contrast", contrast)
        part["hdi_excludes_zero"] = (
            (part["hdi_low"] > 0) | (part["hdi_high"] < 0)
        )
        for rope in ROPE_GRID:
            label = str(rope).replace(".", "p")
            within = within_assessments[(contrast, rope)][
                "effect_metrics"
            ].set_index("global_feature").loc[feature_ids]
            for column in (
                "local_null_probability", "indicator_ess",
                "indicator_mcse_guarded", "local_null_mc_lower",
                "local_null_mc_upper", "bfdr_rank",
                "cumulative_bfdr_at_rank", "bfdr_selected",
            ):
                part[f"within_contrast_{column}_rope_{label}"] = within[
                    column
                ].to_numpy()
            hypothesis_ids = [f"{contrast}||{name}" for name in feature_ids]
            global_metrics = global_metric_lookup[rope].loc[hypothesis_ids]
            for column in (
                "local_null_probability", "indicator_ess",
                "indicator_mcse_guarded", "bfdr_rank",
                "cumulative_bfdr_at_rank", "bfdr_selected",
            ):
                part[f"global_categorical_{column}_rope_{label}"] = global_metrics[
                    column
                ].to_numpy()
            atomic_csv(
                within_assessments[(contrast, rope)]["loo"],
                OUTPUT_DIR / f"loo_{contrast}_rope_{label}.csv",
            )
        result_parts.append(part)
    results = pd.concat(result_parts, ignore_index=True)
    results_path = OUTPUT_DIR / "categorical_feature_results.csv"
    atomic_csv(results, results_path)

    celltype_parts = []
    for contrast, variable in CELLTYPE_EFFECT_VARIABLES.items():
        beta = idata.posterior[variable].transpose("chain", "draw", "celltype")
        part = posterior_summary_frame(beta, "celltype", "global_celltype")
        part.insert(0, "contrast", contrast)
        part["hdi_excludes_zero"] = (
            (part["hdi_low"] > 0) | (part["hdi_high"] < 0)
        )
        celltype_parts.append(part)
    celltype_results = pd.concat(celltype_parts, ignore_index=True)
    celltype_path = OUTPUT_DIR / "categorical_celltype_results.csv"
    atomic_csv(celltype_results, celltype_path)

    descriptive_rows = []
    group_pairs = {
        "overweight_vs_normal": ("overweight", "normal_weight"),
        "obese_vs_normal": ("obese", "normal_weight"),
        "obese_vs_overweight": ("obese", "overweight"),
    }
    for feature, frame in analysis_data.groupby("global_feature", sort=True):
        identity = frame.iloc[0]
        for contrast, (first_group, second_group) in group_pairs.items():
            first = frame.loc[
                frame["bmi_group"].eq(first_group), "score_model"
            ].to_numpy(float)
            second = frame.loc[
                frame["bmi_group"].eq(second_group), "score_model"
            ].to_numpy(float)
            descriptive_rows.append({
                "contrast": contrast,
                "global_feature": feature,
                "source_compartment": identity["source_compartment"],
                "CellType": identity["CellType"],
                "Signature": identity["Signature"],
                "first_group": first_group,
                "second_group": second_group,
                "n_first": len(first),
                "n_second": len(second),
                "hedges_g_unadjusted": hedges_g(first, second),
                "important": "descriptive_only_not_the_adjusted_model_coefficient",
            })
    descriptive = pd.DataFrame(descriptive_rows)
    descriptive_path = OUTPUT_DIR / "descriptive_hedges_g.csv"
    atomic_csv(descriptive, descriptive_path)

    reveal_manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "model_fingerprint": MODEL_FINGERPRINT,
        "notebook_release_sha256": NOTEBOOK_RELEASE["sha256"],
        "reveal_version": REVEAL_VERSION,
        "run_spec": RUN_SPEC,
        "profile": PROFILE,
        "selection_status": "revealed_after_all_gates",
        "selection_revealed": True,
        "selection_reveal_utc": datetime.now(timezone.utc).isoformat(),
        "analysis_lock_sha256": ANALYSIS_LOCK_SHA256,
        "posterior_trace_sha256": sha256_file(TRACE_PATH),
        "diagnostics_sha256": sha256_file(diagnostics_path),
        "ppc_adequacy_sha256": sha256_file(ppc_path),
        "feature_inventory_sha256": sha256_file(
            OUTPUT_DIR / "feature_inventory.csv"
        ),
        "precision_gate_sha256": sha256_file(
            OUTPUT_DIR / "reveal_precision_gate.json"
        ),
        "results_sha256": sha256_file(results_path),
        "celltype_results_sha256": sha256_file(celltype_path),
        "descriptive_hedges_g_sha256": sha256_file(descriptive_path),
        "rope_grid": list(ROPE_GRID),
        "multiplicity_contract": {
            "within_contrast": "reported for compatibility and sensitivity",
            "global_three_contrast": (
                "formal categorical selection within this analysis family"
            ),
        },
        "continuous_primary_results_unchanged": True,
    }
    reveal_manifest_path = OUTPUT_DIR / "selection_reveal_manifest.json"
    atomic_json(reveal_manifest, reveal_manifest_path)
    atomic_json({
        "publication_adequacy_pass": True,
        "convergence_pass": True,
        "ppc_adequacy_pass": True,
        "rope_bfdr_precision_pass": True,
        "selection_reveal_manifest_sha256": sha256_file(reveal_manifest_path),
    }, OUTPUT_DIR / "publication_adequacy.json")
    print(json.dumps(reveal_manifest, indent=2))
    primary_label = str(PRIMARY_ROPE).replace(".", "p")
    display(results.sort_values(
        [f"global_categorical_local_null_probability_rope_{primary_label}",
         "posterior_mean"],
        ascending=[True, False],
    ).head(50))
else:
    print("Result reveal disabled; no categorical effect table was computed.")


In [ ]:
# 11. Validate and aggregate all completed categorical compartments
if RUN_PRIMARY_AGGREGATION:
    aggregate_dir = MODEL_ROOT / "global_primary_categorical_bfdr"
    aggregate_dir.mkdir(parents=True, exist_ok=True)

    def choose_completed_run(spec_name):
        for profile in ("publication_long", "publication"):
            directory = run_directory_for(spec_name, profile)
            required = {
                "results": directory / "categorical_feature_results.csv",
                "celltypes": directory / "categorical_celltype_results.csv",
                "diagnostics": directory / "run_diagnostics.json",
                "ppc": directory / "ppc_adequacy.json",
                "precision": directory / "reveal_precision_gate.json",
                "reveal": directory / "selection_reveal_manifest.json",
                "trace": directory / "posterior_trace.nc",
                "lock": directory / "analysis_lock.json",
            }
            if all(path.exists() for path in required.values()):
                return profile, directory, required
        raise FileNotFoundError(f"No fully revealed publication run for {spec_name}")

    result_parts = []
    celltype_parts = []
    source_records = []
    for spec_name in RUN_SPECS:
        profile, directory, paths = choose_completed_run(spec_name)
        diagnostics = json.loads(paths["diagnostics"].read_text(encoding="utf-8"))
        ppc = json.loads(paths["ppc"].read_text(encoding="utf-8"))
        precision = json.loads(paths["precision"].read_text(encoding="utf-8"))
        reveal = json.loads(paths["reveal"].read_text(encoding="utf-8"))
        checks = {
            "diagnostic_gate": diagnostics.get("current_profile_gate_pass") is True,
            "ppc_gate": ppc.get("ppc_adequacy_pass") is True,
            "precision_gate": precision.get("all_displayed_ropes_pass") is True,
            "reveal_status": reveal.get("selection_revealed") is True,
            "architecture": reveal.get("model_architecture") == MODEL_ARCHITECTURE,
            "release": diagnostics.get("notebook_release_sha256") == NOTEBOOK_RELEASE["sha256"],
            "trace_hash": reveal.get("posterior_trace_sha256") == sha256_file(paths["trace"]),
            "lock_hash": reveal.get("analysis_lock_sha256") == sha256_file(paths["lock"]),
            "diagnostics_hash": reveal.get("diagnostics_sha256") == sha256_file(paths["diagnostics"]),
            "ppc_hash": reveal.get("ppc_adequacy_sha256") == sha256_file(paths["ppc"]),
            "results_hash": reveal.get("results_sha256") == sha256_file(paths["results"]),
            "celltypes_hash": reveal.get("celltype_results_sha256") == sha256_file(paths["celltypes"]),
        }
        failed = [name for name, passed in checks.items() if not passed]
        if failed:
            raise RuntimeError(f"{spec_name} aggregation provenance failed: {failed}")
        frame = pd.read_csv(paths["results"])
        frame.insert(0, "source_profile", profile)
        frame.insert(0, "source_run_spec", spec_name)
        result_parts.append(frame)
        cells = pd.read_csv(paths["celltypes"])
        cells.insert(0, "source_profile", profile)
        cells.insert(0, "source_run_spec", spec_name)
        celltype_parts.append(cells)
        source_records.append({
            "run_spec": spec_name,
            "role": RUN_SPECS[spec_name]["role"],
            "profile": profile,
            "directory": str(directory),
            "results_sha256": sha256_file(paths["results"]),
            "celltype_results_sha256": sha256_file(paths["celltypes"]),
            "reveal_manifest_sha256": sha256_file(paths["reveal"]),
        })

    combined = pd.concat(result_parts, ignore_index=True)
    combined_celltypes = pd.concat(celltype_parts, ignore_index=True)
    primary_mask = combined["source_run_spec"].isin(
        [name for name, spec in RUN_SPECS.items() if spec["role"] == "primary"]
    )
    for rope in ROPE_GRID:
        label = str(rope).replace(".", "p")
        null_column = f"global_categorical_local_null_probability_rope_{label}"
        selection_column = f"global_primary_bfdr_selected_rope_{label}"
        combined[selection_column] = False
        primary = combined.loc[primary_mask].copy()
        identifiers = (
            primary["source_run_spec"].astype(str) + "||"
            + primary["contrast"].astype(str) + "||"
            + primary["global_feature"].astype(str)
        ).to_numpy()
        selected, _, _, _ = deterministic_bfdr(
            primary[null_column].to_numpy(float), identifiers, Q_LEVEL
        )
        combined.loc[primary.index, selection_column] = selected
    combined_path = aggregate_dir / "all_compartments_categorical_results.csv"
    celltype_path = aggregate_dir / "all_compartments_categorical_celltype_results.csv"
    atomic_csv(combined, combined_path)
    atomic_csv(combined_celltypes, celltype_path)
    manifest = {
        "analysis_version": ANALYSIS_VERSION,
        "model_architecture": MODEL_ARCHITECTURE,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "primary_families": ["immune_coarse", "nonimmune"],
        "secondary_family": "immune_fine",
        "reported_contrasts": list(REPORTED_CONTRASTS),
        "global_primary_multiplicity_scope": (
            "all three contrasts and all pathway hypotheses in immune-coarse "
            "and non-immune primary families"
        ),
        "q_level": Q_LEVEL,
        "rope_grid": list(ROPE_GRID),
        "sources": source_records,
        "combined_results_sha256": sha256_file(combined_path),
        "combined_celltype_results_sha256": sha256_file(celltype_path),
        "important": (
            "The continuous analysis remains primary. This aggregation is the "
            "formal multiplicity-controlled categorical sensitivity summary."
        ),
    }
    atomic_json(manifest, aggregate_dir / "aggregate_manifest.json")
    print(json.dumps(manifest, indent=2))
    print("Saved:", combined_path)
else:
    print("Categorical aggregation disabled.")
